https://tonikph-my.sharepoint.com/:p:/g/personal/bbanik_tonikbank_com/IQAvsct_VJYKRoYRQISh8jy1AWHNdHcuDqk9bIfwQjpiW6M?e=wvYx82

# Import Packages

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta, date
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
import re
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"
# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)
sns.set_style('whitegrid')

# CONFIGURATION

In [2]:
# GCP credentials — use ADC JSON or set GOOGLE_APPLICATION_CREDENTIALS env var
CREDENTIALS_PATH = r"C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = CREDENTIALS_PATH

os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

PROJECT_ID = "prj-prod-dataplatform"
GS_BUCKET = "prod-asia-southeast1-tonik-aiml-workspace"

CLOUDPATH = "DC/Tendo_Model_Monitoring_Data"
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Output folder — Power BI reads CSVs from here
OUTPUT_DIR = r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data"


# Functions

In [3]:
def define_cat_features(columns, cat_vars):
    return list(set(cat_vars).intersection(columns))

def generate_bucket_url(filename, bucket_name):

    return f"gs://{bucket_name}/{filename}"


def save_to_gcs(data, filename, bucket_name):
    """
    Save data to Google Cloud Storage bucket.

    :param data: The data to save. Can be a string, bytes, or a file-like object.
    :param filename: The name of the file to save in the bucket.
    :param bucket_name: The name of the GCS bucket. Defaults to 'ABC'.
    :return: The public URL of the saved file.
    """
    # Create a client
    client = storage.Client()

    # Get the bucket
    bucket = client.bucket(bucket_name)

    # Create a blob (file) in the bucket
    blob = bucket.blob(filename)

    # Determine the content type and upload accordingly
    if isinstance(data, str):
        blob.upload_from_string(data)
    elif isinstance(data, bytes):
        blob.upload_from_string(data, content_type='application/octet-stream')
    elif hasattr(data, 'read'):  # File-like object
        blob.upload_from_file(data)
    else:
        raise ValueError("Unsupported data type. Please provide a string, bytes, or file-like object.")


    print(f"File {filename} uploaded to {bucket_name}.")
    return blob.public_url


def load_artifact_from_gcs(filename, bucket_name):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Download model
    artifact_bytes = blob.download_as_bytes()
    artifact = pickle.loads(artifact_bytes)
    print(f"Model loaded from gs://{bucket_name}/{filename}")

    return artifact


def load_from_gcs(filename, bucket_name, output_type='bytes'):
    """
    Load data from Google Cloud Storage bucket with flexible output formats.

    :param filename: The name of the file to load from the bucket.
    :param bucket_name: The name of the GCS bucket.
    :param output_type: The desired output format. Can be 'bytes', 'string', 'pickle', or 'file'.
                       'bytes': Returns raw bytes
                       'string': Returns decoded string
                       'pickle': Returns unpickled Python object
                       'file': Returns a file-like object for streaming
    :return: The loaded data in the specified format.
    """
    import pickle
    import io

    # Create a client
    client = storage.Client()

    # Get the bucket and blob
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Handle different output types
    if output_type == 'bytes':
        return blob.download_as_bytes()

    elif output_type == 'string':
        return blob.download_as_string().decode('utf-8')

    elif output_type == 'pickle':
        pickled_data = blob.download_as_bytes()
        return pickle.loads(pickled_data)

    elif output_type == 'file':
        file_obj = io.BytesIO()
        blob.download_to_file(file_obj)
        file_obj.seek(0)  # Reset file pointer to beginning
        return file_obj

    else:
        raise ValueError("Unsupported output_type. Must be one of: 'bytes', 'string', 'pickle', 'file'")

def load_artifacts_logreg(exp_number):

    model_filename = f'Oleh/tendo/experiments/{exp_number}/model.pkl'
    model = load_artifact_from_gcs(model_filename, GS_BUCKET)

    feature_filename = f'Oleh/tendo/experiments/{exp_number}/features.pkl'
    features = load_artifact_from_gcs(feature_filename, GS_BUCKET)

    scaler_filename = f'Oleh/tendo/experiments/{exp_number}/scaler.pkl'
    scaler = load_artifact_from_gcs(scaler_filename, GS_BUCKET)

    return model, features, scaler

def save_artifact_to_gcs(artifact, filename, bucket_name):
    """Saves the Cox model to Google Cloud Storage."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Serialize artifact
    artifact_bytes = pickle.dumps(artifact)

    # Upload to GCS
    blob.upload_from_string(artifact_bytes)
    print(f"Artifact saved to gs://{bucket_name}/{filename}")


import pickle
import io
from google.cloud import storage
def save_pickle_to_gcs(data, bucket_name, destination_blob_name):
    """
    Save any Python object as a pickle file to Google Cloud Storage
    
    Args:
        data: The Python object to pickle (DataFrame, dict, list, etc.)
        bucket_name: Name of the GCS bucket
        destination_blob_name: Path/filename in the bucket
    """
    # Initialize the GCS client
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    
    # Serialize the data to pickle format in memory
    pickle_buffer = io.BytesIO()
    pickle.dump(data, pickle_buffer)
    pickle_buffer.seek(0)
    
    # Upload the pickle data to GCS
    blob.upload_from_file(pickle_buffer, content_type='application/octet-stream')
    print(f"Pickle file uploaded to gs://{bucket_name}/{destination_blob_name}")

In [4]:
import pandas as pd
from google.cloud import bigquery
from sklearn.metrics import roc_auc_score
from typing import Dict

def calculate_gini_for_table(
    data: pd.DataFrame,
    date_column: str,
    score_column: str,
    target_column: str,
    data_periods_dict: Dict,
    weights_col: str = None
):
    """
    Calculate Gini coefficient for different time periods.

    Args:
        project_id: BigQuery project ID
        table_name: Full table name (dataset.table)
        date_column: Name of the date column
        score_column: Name of the score column
        target_column: Name of the target column
        target_maturity_column: Name of the target maturity column
        data_periods_dict: Dictionary with periods, e.g.:
            {'Train': {'start': '2024-01-01', 'end': '2025-01-31'},
             'Test': {'start': '2025-02-01', 'end': '2025-12-31'}}

    Returns:
        pandas.DataFrame: Table with Gini coefficients for each period
    """
    dt = data[data[target_column].notna()].copy()

    # Convert date column to datetime and extract just the date part
    dt[date_column] = pd.to_datetime(dt[date_column]).dt.date

    # Initialize results
    gini_results = []

    print("Gini Coefficient Results:")
    print("=" * 50)

    # Calculate Gini for each period
    for period_name, period_info in data_periods_dict.items():
        start_date = pd.to_datetime(period_info['start']).date()
        end_date = pd.to_datetime(period_info['end']).date()

        # Filter data for the current period
        period_mask = (dt[date_column] >= start_date) & (dt[date_column] <= end_date)
        period_data = dt[period_mask].copy()

        sample_size = period_data['ee_customer_id'].nunique()
        bad_rate = 100*(1 - period_data[['ee_customer_id',target_column]].drop_duplicates()[target_column].sum() / sample_size)

        if len(period_data) == 0:
            print(f"{period_name}: No data available for period {start_date} to {end_date}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': 0,
                'Bad Rate': np.nan,
                'Gini_Coefficient': None
            })
            continue

        # Check if we have both classes (0 and 1) in target
        unique_targets = period_data[target_column].unique()
        if len(unique_targets) < 2:
            print(f"{period_name}: Only one class present in target variable. Cannot calculate Gini.")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })
            continue

        # Calculate Gini coefficient
        try:
            if weights_col:
                auc = roc_auc_score(period_data[target_column], period_data[score_column], sample_weight=period_data[weights_col])
                gini = 2 * auc - 1
            else:
                auc = roc_auc_score(period_data[target_column], period_data[score_column])
                gini = 2 * auc - 1

            print(f"{period_name}: {round(gini, 4)} (Sample size: {len(period_data):,})")

            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': round(gini, 4)
            })

        except Exception as e:
            print(f"{period_name}: Error calculating Gini - {str(e)}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })

    # Create results DataFrame
    results_df = pd.DataFrame(gini_results)

    print("\n" + "=" * 50)
    print("Summary Table:")
    print(results_df.to_string(index=False))

    return results_df

# Data loading

## PROD API

In [5]:
# PROD API
sql_query_prod_api = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api`
"""

dt_prod_api = client.query(sql_query_prod_api).to_dataframe()

In [6]:
print(f"The shape of dt_prod_api is :\t{dt_prod_api.shape}")

The shape of dt_prod_api is :	(165391, 6)


In [7]:
dt_prod_api.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
103180,1512965,2025-08-10T05:53:06.295146,10-12,10,0.527099,B
116026,1492859,2025-07-22T03:42:46.792693,9,9,0.646942,A
155866,1432823,2025-09-13T07:03:55.530918,4,4,0.441632,F


## PROD BATCH

In [8]:
# PROD BATCH
sql_query_prod_batch = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`
"""

dt_prod_batch = client.query(sql_query_prod_batch).to_dataframe()
print(f"The shape of dt_prod_batch is :\t{dt_prod_batch.shape}")

The shape of dt_prod_batch is :	(407568, 6)


In [9]:
dt_prod_batch.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
404505,1298577,2025-08-07,Average,9,0.582339,A
230767,1619287,2026-02-16,Very low,12+,0.637825,B
171099,1293254,2025-10-29,High,6,0.448156,D


## OOP Latest targets

In [10]:
# OOP Latest targets
sql_query_oop_targets = """
SELECT
  user_id as ee_customer_id,
  target as oop_target
FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
WHERE target_maturity_flag = 1
"""

dt_oop_targets = client.query(sql_query_oop_targets).to_dataframe()
print(f"The shape of dt_oop_targets is :\t{dt_oop_targets.shape}")

The shape of dt_oop_targets is :	(36336, 2)


In [11]:
dt_oop_targets.groupby('oop_target')['ee_customer_id'].nunique()

oop_target
0    28321
1     8015
Name: ee_customer_id, dtype: int64

## tendo_scorecard_features_data

In [12]:
dt = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-03-2026`;""").to_dataframe()

print(f"The shape of the tendo scorecard feature data is: {dt.shape}")

The shape of the tendo scorecard feature data is: (4168379, 100)


In [13]:
dt.head()

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag
0,1215362,christopher.alluad@gmail.com,+63 (995) 098 2456,Christopher,Agraan,Alluad,1987-09-02,Male,1,1,2,2,None,None,2,NCR - NATIONAL CAPITAL REGION,None,PLAINVIEW,"Raymond Tower, 615, Boni Avenue, Mandaluyong, ...",None,None,Municipal City Hall,None,regular,Single,Payslip,true,2024-06-27 10:35:41+00:00,2023-03-20,Risk,None,"110,Resigned",None,None,Accountant,Permanent Job (Private sector),None,2025-10-23 19:07:36+00:00,2025-10-23,employer,Frozen,123,118,146,117,504,B,10182.000000000,1.000000000,TOA Global Manila Inc.,None,"[""5"",""20""]",1,-1,None,None,2.5,8,IN_PROGRESS,2023-09-15 16:06:11.000000,NaT,2023-09-15 16:06:11+00:00,2025-08-07 17:05:03+00:00,1300,41500,36000,1,280,50,1,1,1,1,1,1,1,1,3525229,Tendo,Approved/Disbursed,PH_BPI,14400.0,4320.0,12,2025-05-12 15:23:53+00:00,0,None,0,0,0,0,2025-06-05,0.0,0.0,0.0,0.0,1,1,1,1
1,1178527,krystel@cloudcfo.ph,+63 (926) 005 9599,Krystel Joy,None,Macarilao,1996-03-31,Female,1,1,2,2,None,None,2,NCR - NATIONAL CAPITAL REGION,None,CUPANG,744 ML Quezon St,None,None,Goodhealth Pharmacy,None,probationary,Single,None,true,2024-05-17 17:52:14+00:00,2024-02-26,Normal,None,None,None,None,Senior Accountant,Permanent Job (Private sector),None,NaT,None,employer,Not Frozen,98,120,146,117,481,C,10204.000000000,None,CLOUDCFO INC.,None,"[""15"",""30""]",1,7,None,None,3.0,12,IN_PROGRESS,2024-01-04 17:57:00.000000,NaT,2024-01-04 17:57:00+00:00,2025-03-17 18:45:33+00:00,10000,35000,31000,1,280,50,1,1,1,1,0,0,1,1,1503432,Tendo,Approved/Disbursed,PH_BPI,15500.0,1395.0,3,2024-05-18 11:10:17+00:00,1,2024-08-30,0,0,0,0,2024-06-15,0.0,0.0,0.0,0.0,1,1,1,1
2,1076394,kinmark.dano@amadeus.com,+63 (956) 791 6537,KIN MARK,MORALES,DANO,1998-11-20,Male,1,1,2,2,None,None,2,NCR - NATIONAL CAPITAL REGION,None,WESTERN BICUTAN,30C ROOM G CHAMPACA STREET,None,None,NEAR CHAMPACA HALF COURT,None,regular,Single,Payslip,true,2023-06-29 15:35:32+00:00,2022-03-01,Risk,None,None,None,None,QUEUES HANDLING ASSOCIATE,Permanent Job (Private sector),None,NaT,None,employer,Frozen,123,118,118,117,476,D,10163

In [14]:
dt.columns.values

array(['ee_customer_id', 'ee_email', 'ee_phone_number', 'ee_firstname',
       'ee_middlename', 'ee_lastname', 'ee_birthdate', 'ee_gender',
       'ee_email_verified_flag', 'ee_telephone_verified_flag',
       'ee_id_verified_flag', 'ee_income_verified_flag',
       'ee_morning_time_contact_time', 'ee_afternoon_time_contact_time',
       'ee_account_activated_flag', 'ee_region_name', 'ee_city_name',
       'ee_barangay', 'ee_address_line_1', 'ee_address_line_2',
       'ee_postal_code', 'ee_landmark', 'ee_residing_date',
       'ee_employment_status', 'ee_civil_status', 'ee_kyc_doc_name',
       'ee_is_citizen_flag', 'ee_onboarding_date', 'ee_employment_date',
       'ee_fraud_status', 'ee_job_type', 'ee_comment', 'ee_department',
       'ee_recommended_ir', 'ee_job_title', 'ee_employment_type',
       'ee_nature_of_work', 'ee_permanent_freeze_date',
       'ee_resignation_date', 'ee_product_type', 'ee_frozen_status',
       'cust_risk_employer_time_score_v1', 'cust_risk_gender_score_v

In [15]:
dt_raw = dt.copy()
dt_raw["ee_customer_id"] = dt_raw["ee_customer_id"].astype(str)
dt_raw["ee_onboarding_date"] = pd.to_datetime(dt_raw["ee_onboarding_date"]).dt.tz_localize(None)
print(f"The shape of the dt_raw a copy of dt is:\t{dt_raw.shape}")
# Keep only the two columns we need from dt_raw to avoid any future collision
dt_raw_slim = dt_raw[["ee_customer_id", "ee_onboarding_date"]].drop_duplicates()
print(f"The shape of the dt_raw_slim after droping the duplicates is:\t{dt_raw_slim.shape}")

The shape of the dt_raw a copy of dt is:	(4168379, 100)
The shape of the dt_raw_slim after droping the duplicates is:	(1742823, 2)


In [16]:
dd.query("""
    SELECT ee_customer_id, count(ee_customer_id) as cnt 
    FROM dt_raw 
    GROUP BY 1 
    HAVING count(ee_customer_id) > 1
    order by 2 desc;
""").to_df()


,ee_customer_id,cnt
0,1065228,1370
1,949503,718
2,1123001,712
3,1094726,698
4,529907,590
...,...,...
239837,1267761,2
239838,1125116,2
239839,1180636,2
239840,1214752,2


In [17]:
dt_raw_slim.sample(3)

,ee_customer_id,ee_onboarding_date
1392554,1078003,NaT
2524510,1661468,2025-12-23 13:40:16
725567,313302,2021-04-03 11:56:33


## tendo_scorecard_loan_repayment_data

In [18]:
dt_repayments = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard repayment data is: {dt_repayments.shape}")

The shape of the tendo scorecard repayment data is: (16086906, 15)


In [19]:
dt_repayments.sample(3)

,user_id,loan_id,installment_number,vendor_id,subfamily,repaid_date,due_date,due_principal,due_interest,due_amount,paid_principal,paid_interest,paid_amount,outstanding_balance,DPD
13658980,1210717,3391736,7,TPSD,COPA,2025-08-19,2025-08-10,49.789,29.541,79.33,49.789,29.541,79.33,1077.076,9
15579687,1378627,3980665,6,TPSD,COPA,2025-11-21,2025-10-30,243.011,5.119,248.13,243.011,5.119,248.13,0.000,22
13950860,1176597,3806517,12,TPSD,COPA,2026-01-20,2026-01-10,179.753,6.247,186.00,179.753,6.247,186.00,0.000,10


## Resignation code

In [20]:
def to_date_str(val) -> str | None:
    # missing
    if val is None or pd.isna(val):
        return None

    # already a string
    if isinstance(val, str):
        return val.strip()

    # datetime-like (Timestamp, datetime, date, numpy datetime64)
    if isinstance(val, (pd.Timestamp, datetime, date, np.datetime64)):
        return pd.Timestamp(val).strftime("%Y-%m-%d")

    # fallback (numbers, objects, etc.)
    return str(val).strip()

def validate_and_convert_dates(df, column_name, output_tag_col, output_date_col,
                                min_date='1990-01-01',
                                max_date='2025-10-01',
                                min_date_col=None,
                                max_date_col=None):
    """
    Validates dates and creates tags with converted dates.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with date column
    column_name : str
        Name of the date column to validate
    min_date : str
        Default minimum date threshold (default: '1990-01-01')
    max_date : str
        Default maximum date threshold (default: '2025-10-01')
    min_date_col : str, optional
        Column name with per-row minimum date thresholds (overrides min_date when not null)
    max_date_col : str, optional
        Column name with per-row maximum date thresholds (overrides max_date when not null)

    Tags:
    - 'valid': Already in correct format and within range
    - 'convertible': Can be fixed (missing day, wrong format) and within range
    - 'invalid': Cannot be converted or outside valid range
    - 'null': NULL/NaN/empty values

    Returns DataFrame with:
    - date_tag: validation tag
    - date_converted: standardized date in YYYY-MM-DD format or None
    """

    default_min_threshold = pd.Timestamp(min_date)
    default_max_threshold = pd.Timestamp(max_date)

    def process_date(row):
        """Process a single date value with per-row thresholds"""

        val = row[column_name]

        # Determine thresholds for this row
        # Use per-row threshold if available and not null, otherwise use default
        if min_date_col and min_date_col in df.columns and pd.notna(row[min_date_col]):
            min_threshold = pd.Timestamp(row[min_date_col])
            # Remove timezone info if present to allow comparison
            if min_threshold.tzinfo is not None:
                min_threshold = min_threshold.tz_localize(None)
        else:
            min_threshold = default_min_threshold

        if max_date_col and max_date_col in df.columns and pd.notna(row[max_date_col]):
            max_threshold = pd.Timestamp(row[max_date_col])
            # Remove timezone info if present to allow comparison
            if max_threshold.tzinfo is not None:
                max_threshold = max_threshold.tz_localize(None)
        else:
            max_threshold = default_max_threshold

        # Handle NULL values
        if pd.isna(val):
            return 'null', None

        date_str = to_date_str(val)

        # Handle empty or 'nan' strings
        if date_str == '' or date_str.lower() == 'nan':
            return 'null', None

        # Check for invalid year pattern (must start with 19 or 20)
        year_match = re.search(r'(\d{4})', date_str)
        if year_match:
            year_str = year_match.group(1)
            first_digit = year_str[0]
            second_digit = year_str[1]

            # Year must start with 1 or 2
            if first_digit not in ['1', '2']:
                return 'invalid', None

            # If starts with 1, second digit must be 9 (1900-1999)
            if first_digit == '1' and second_digit != '9':
                return 'invalid', None

            # If starts with 2, second digit must be 0 (2000-2099)
            if first_digit == '2' and second_digit != '0':
                return 'invalid', None

        # Try to parse and convert the date
        converted_date = None
        needs_conversion = False

        # Pattern 1: YYYY-MM-DD (standard format with various separators)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month, day = match.groups()
                needs_conversion = False  # Reset for each pattern

                # Store original to check if padding needed
                original_month_len = len(month)
                original_day_len = len(day)

                # Pad month and day if needed
                month = month.zfill(2)
                day = day.zfill(2)

                # Check if padding was needed
                if original_month_len == 1 or original_day_len == 1:
                    needs_conversion = True

                # DON'T swap - the input format is already YYYY-MM-DD
                # (We only swap for DD-MM-YYYY format in Pattern 4)

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    # Different separator means needs conversion
                    if sep != '-':
                        needs_conversion = True

                    return 'convertible' if needs_conversion else 'valid', converted_date
                except:
                    return 'invalid', None

        # Pattern 2: YYYY-MM (missing day)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month = match.groups()
                month = month.zfill(2)
                day = '01'  # Default to first day of month

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # Pattern 3: YYYYMMDD (no separator)
        if re.match(r'^\d{8}$', date_str):
            year = date_str[0:4]
            month = date_str[4:6]
            day = date_str[6:8]

            try:
                parsed = pd.Timestamp(f"{year}-{month}-{day}")
                converted_date = f"{year}-{month}-{day}"

                # Check if within valid range
                if parsed < min_threshold or parsed > max_threshold:
                    return 'invalid', None

                return 'convertible', converted_date
            except:
                return 'invalid', None

        # Pattern 4: DD-MM-YYYY or MM-DD-YYYY (need to detect which)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{4}})$'
            match = re.match(pattern, date_str)
            if match:
                part1, part2, year = match.groups()
                part1 = part1.zfill(2)
                part2 = part2.zfill(2)

                # Determine if DD-MM-YYYY or MM-DD-YYYY
                # If part1 > 12, it must be day, so DD-MM-YYYY
                if int(part1) > 12:
                    day, month = part1, part2
                # If part2 > 12, it must be day, so MM-DD-YYYY
                elif int(part2) > 12:
                    month, day = part1, part2
                # Ambiguous case: assume MM-DD-YYYY (common US format)
                else:
                    # Could be either, default to MM-DD-YYYY but mark as needing review
                    month, day = part1, part2

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # If nothing matched, it's invalid
        return 'invalid', None

    # Apply processing to all rows
    results = df.apply(process_date, axis=1)

    df_result = pd.DataFrame(index=df.index)
    df_result[output_tag_col] = results.apply(lambda x: x[0])
    df_result[output_date_col] = pd.to_datetime(results.apply(lambda x: x[1]))

    return df_result

# Filtering
dt = dt[dt['ee_product_type'] == 'employer']
dt = dt[dt['ee_onboarding_date'].notna()].reset_index(drop=True)

# loan table preparation
str_columns = ['ee_customer_id', 'ln_loan_id'] 

missing_values = ['<NA>', 'None', 'NaN', 'nan', '', 'null', 'NULL']
dt[str_columns] = dt[str_columns].replace(missing_values, np.nan).astype('string')

localize_date_cols = ['ee_permanent_freeze_date','ee_onboarding_date']
for col in localize_date_cols:
    print(col)
    dt[col] = pd.to_datetime(dt[col]).dt.tz_localize(None)

date_col_format = ['ee_resignation_date']
for date_col in date_col_format:
    dt[date_col] = pd.to_datetime(dt[date_col], format='%Y-%m-%d', errors='coerce')

# repayment table preparation
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')



resignation_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_resignation_date']].first(),
    column_name='ee_resignation_date',
    output_tag_col='ee_resignation_date_tag',
    output_date_col='ee_resignation_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

permanent_freeze_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_permanent_freeze_date']].first(),
    column_name='ee_permanent_freeze_date',
    output_tag_col='ee_permanent_freeze_date_tag',
    output_date_col='ee_permanent_freeze_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

resignation_dates = resignation_dates.merge(
        permanent_freeze_dates['ee_permanent_freeze_date_validated'],
        how='left',
        on='ee_customer_id'
)

resignation_dates['ee_resignation_date_correct'] = resignation_dates.loc[:, 'ee_resignation_date_validated'].fillna(resignation_dates['ee_permanent_freeze_date_validated'])

dt = dt.merge(
    resignation_dates['ee_resignation_date_correct'].reset_index(),
    how='left',on='ee_customer_id'
)

dt_repayments['vendor_id_shifted'] = dt_repayments.groupby('loan_id')[['vendor_id']].shift(-1)
dt_repayments['repaid_date_shifted'] = dt_repayments.groupby('loan_id')[['repaid_date']].shift(-1)

resignation_dates_repayments = (
    dt_repayments[['user_id','loan_id','vendor_id','vendor_id_shifted','repaid_date']]
    .query('vendor_id == "TPSD"')
    .fillna('TPSD')
    .query('vendor_id != vendor_id_shifted & vendor_id_shifted == "TPFP"')
    .assign(resignation_date_fp_new = lambda x: x['repaid_date'] + pd.DateOffset(months=1))
    .groupby('user_id')[['resignation_date_fp_new']]
    .max()
    .reset_index()
)

dt_resignation = dt.merge(resignation_dates_repayments, how='left', left_on='ee_customer_id', right_on='user_id')
dt_resignation['ee_resignation_date_correct'] = dt_resignation['ee_resignation_date_correct'].fillna(dt_resignation['resignation_date_fp_new'])

dt_res = dt_resignation[['ee_customer_id','ee_resignation_date_correct']].drop_duplicates() 

ee_permanent_freeze_date
ee_onboarding_date


In [21]:
dt_resignation.sample(3)

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag,ee_resignation_date_correct,user_id,resignation_date_fp_new
614500,1236398,saladorojon@gmail.com,+63 (927) 605 1624,Rojon Jr,Poticar,Salado,2002-08-10,Male,1,1,2,2,None,None,2,None,None,None,Sitio sool 2 Brgy 2,None,None,Purok Ugyon,None,regular,Single,Barangay Certificate w/ picture,true,2024-07-23 12:30:51,2023-03-13,Normal,None,None,None,None,Costumer Service Representative,Permanent Job (Private sector),None,NaT,NaT,employer,Not Frozen,123,118,119,117,477,D,10239.000000000,None,Ubiquity,None,"[""10"",""25""]",1,-1,None,None,3.0,8,IN_PROGRESS,2024-04-26 18:01:21.000000,NaT,2024-04-26 18:01:21+00:00,2025-01-22 16:31:58+00:00,2000,17500,16500,1,187,25,1,1,1,1,0,0,0,0,3656729,Tendo,Approved/Disbursed,PH_UBP,9000.0,2160.0,8,2025-06-02 16:20:41+00:00,1,2026-02-10,0,0,0,0,2025-06-25,0.0,0.0,0.0,0.0,1,1,1,1,NaT,NaN,NaT
331678,1219792,domingomarkvincent5@gmail.com,+63 (927) 800 5986,Mark Vincent,Salinas,Domingo,1997-10-14,Male,1,1,2,2,None,None,2,None,None,None,PUROK 5 SINILIAN 2nd,None,None,Near Elementary school,None,regular,Single,Barangay Certificate w/ picture,true,2024-06-26 11:42:54,None,Normal,None,None,None,None,Customer Support Engineer,Permanent Job (Private sector),None,NaT,NaT,employer,Frozen,123,118,146,117,504,B,10240.000000000,None,OUTSOURCED QUALITY ASSURED SERVICES INC.,None,"[""5"",""20""]",1,4,None,None,3.5,8,IN_PROGRESS,2024-05-07 21:03:11.000000,NaT,2024-05-07 21:03:11+00:00,2025-06-13 16:59:57+00:00,7000,79500,65000,1,280,45,1,1,1,1,0,0,0,0,7368807,Tendo,Approved/Disbursed,PH_SEC,28000.0,7280.0,8,2026-02-11 17:41:21+00:00,0,None,0,0,0,0,2026-03-05,0.0,0.0,0.0,0.0,0,0,0,0,NaT,NaN,NaT
2284617,1592608,dhavidzandal@gmail.com,+63 (929) 315 7690,david,fuentes,andal,1984-12-02,Male,1,1,2,2,None,None,2,Region IV-A,None,BARANGAY II (POB.),2146 A Mabini Street,None,None,Aglipay church,None,regular,Single,None,true,2025-10-22 10:45:44,2021-02-17,None,None,None,None,None,Associate Coach,Permanent Job (Private sector),26,NaT,NaT,employer,Not Frozen,183,118,146,117,564,A,10301.000000000,None,iQor

In [22]:
print(f"The shape of the dt_resignation is:\{dt_resignation.shape}")

The shape of the dt_resignation is:\(2509446, 103)


<string>:1: SyntaxWarning: invalid escape sequence '\{'
<>:1: SyntaxWarning: invalid escape sequence '\{'
<string>:1: SyntaxWarning: invalid escape sequence '\{'
<>:1: SyntaxWarning: invalid escape sequence '\{'
C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_41656\467682833.py:1: SyntaxWarning: invalid escape sequence '\{'
  print(f"The shape of the dt_resignation is:\{dt_resignation.shape}")


In [23]:
# Your modified code
bucket_name = GS_BUCKET
pickle_filename = f"resignation_data_{CURRENT_DATE}"

# Construct the new filename with .pkl extension
data_filename = f"{pickle_filename}.pkl"
print(data_filename)

destination_blob_name = f"{CLOUDPATH}/{data_filename}"

# Save the DataFrame (or any object) as pickle to GCS
save_pickle_to_gcs(dt_resignation[['ee_customer_id','ee_resignation_date_correct']], bucket_name, destination_blob_name)



resignation_data_20260331.pkl
Pickle file uploaded to gs://prod-asia-southeast1-tonik-aiml-workspace/DC/Tendo_Model_Monitoring_Data/resignation_data_20260331.pkl


## Outstanding balance code

In [24]:
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

# converting loan_id and user_id to str in order to be aligned with loan data
dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')

# merging with loan data to get columns for actual osbal calculations
dt_repayments = dt_repayments.merge(dt[['ln_loan_id','ln_original_principal', 'ln_orig_interest_fees']].drop_duplicates(),
                                    how='left', left_on='loan_id', right_on='ln_loan_id')

# converting dates
dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

# Calculate cumulative paid amount per loan
dt_repayments['cumulative_paid_principal'] = dt_repayments.groupby('loan_id')['paid_principal'].cumsum()

# Calculate osbal_after directly from loan_amount - cumulative_paid
dt_repayments['osbal_after'] = dt_repayments['ln_original_principal'] - dt_repayments['cumulative_paid_principal']

# Calculate osbal_before by shifting osbal_after and using loan_amount for first payment
dt_repayments['osbal_before'] = dt_repayments.groupby('loan_id')['osbal_after'].shift(1)

# Fill first payment osbal_before with loan_amount
mask = dt_repayments['osbal_before'].isna()
dt_repayments.loc[mask, 'osbal_before'] = dt_repayments.loc[mask, 'ln_original_principal']

In [25]:
dt_repayments.sample(2)

,user_id,loan_id,installment_number,vendor_id,subfamily,repaid_date,due_date,due_principal,due_interest,due_amount,paid_principal,paid_interest,paid_amount,outstanding_balance,DPD,vendor_id_shifted,repaid_date_shifted,ln_loan_id,ln_original_principal,ln_orig_interest_fees,cumulative_paid_principal,osbal_after,osbal_before
2968551,1140682,1782533,6,TPSD,COPA,2024-11-15,2024-11-15,543.943,11.457,555.4,543.943,11.457,555.4,0.000,0,NaN,NaT,1782533,3100.0,232.5,3099.999,0.001,543.944
12950724,1339207,4734380,1,TPSD,COPA,2025-11-24,2025-11-25,878.924,283.576,1162.5,878.924,283.576,1162.5,9121.076,-1,TPSD,2025-12-11,4734380,10000.0,1625.0,878.924,9121.076,10000.000


In [26]:
dt_raw.sample(3)

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag
4099274,1558009,namatakristinemae@gmail.com,+63 (953) 099 8362,Kristine Mae,Sumayang,Namata,2005-07-10,Female,0,1,2,2,None,None,2,Region VII,None,IBABAO-ESTANCIA,Ibabao Estancia Mandue City Cebu,None,None,Barz Apartment,None,regular,Single,None,true,2025-09-21 14:11:56,2024-08-05,Risk,None,"110,Resigned",None,None,Customer Service Representatives,Permanent Job (Private sector),26,2025-12-26 17:00:39+00:00,2025-12-26,employer,Frozen,123,120,110,117,470,D,10312.000000000,None,Foundever,None,"[""13"",""28""]",1,-1,None,None,3.5,8,IN_PROGRESS,None,NaT,2025-08-19 18:17:54+00:00,2026-01-19 12:25:13+00:00,1500,18500,15000,1,188,30,1,1,1,1,1,1,1,1,4998378,Tendo,Approved/Disbursed,PH_GCASH,750.0,157.5,6,2025-11-17 20:06:50+00:00,1,2026-01-28,0,1,1,1,2025-12-13,0.0,643.058,0.000,0.0,1,1,1,0
2928222,980215,kelkel.r.sahagun@gmail.com,+63 (936) 674 4399,xzekiel,Ronquillo,Sahagun,1990-08-09,Male,1,1,2,2,None,None,2,Region III,None,MANIBAUG PARALAYA,18 yakal st aguas subd,None,None,st.ignatius parish church,None,regular,Married,Barangay Certificate w/ picture,true,2023-02-10 14:05:09,None,Risk,None,None,None,2021-11-10,Senior team member,Permanent Job (Private sector),None,NaT,None,employer,Frozen,123,118,118,127,486,C,10103.000000000,None,Connext Global Solutions Inc.,None,"[""10"",""25""]",1,14,"Unit 1f-6, Business center 9 Philexcel busines...",496,4.0,8,IN_PROGRESS,2021-11-09 01:29:53.000000,NaT,2021-11-09 01:29:53+00:00,2025-08-28 01:33:18+00:00,3000,40000,32000,1,165,60,1,1,1,1,0,0,0,0,837924,Tendo,Approved/Disbursed,PH_GCASH,1350.0,324.0,6,2023-07-11 02:55:27+00:00,1,2024-01-25,0,0,0,0,2023-08-10,0.0,0.000,0.000,0.0,1,1,1,1
2999529,1047022,rn9bk7p584@privaterelay.appleid.com,+63 (910) 935 7088,John Lester,Lagrama,Relativo,1994-03-18,Male,1,1,2,2,None,None,2,None,None,None,Blk 8 Lot 13 GRSV,None,None,Before Meralco Laguna,None,regular,Married,Barangay Certificate w/ picture,true,2024-04-20 15:19:15,2020-11-02,Risk,None,"110,Resigned",None,None,GSD - IT,Permanent Job (Private sector),None,2025-07-07 11:48:03+0

# OOP

In [27]:
# BS OOP new
sql_query_oop_new = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
"""

dt_bs_oop_new = client.query(sql_query_oop_new).to_dataframe()
print(f"The shape of the dataframe after reading tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal table is :\t {dt_bs_oop_new.shape}")

The shape of the dataframe after reading tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal table is :	 (169297, 7)


In [28]:
dt_bs_oop_new.sample(3)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
26161,1178675,2024-05-01,0.0,0.567186,43027.212,43027.212,43027.212
101872,1615394,2025-12-01,NaN,0.537874,NaN,NaN,37715.116
136198,1459666,2025-07-01,NaN,0.589554,NaN,NaN,45992.925


In [29]:
# BS OOP existing
sql_query_oop_existing = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop`
"""

dt_bs_oop_ex = client.query(sql_query_oop_existing).to_dataframe()
print(f"The shape of dataframe after reading table `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop` is:\t{dt_bs_oop_ex.shape}")

The shape of dataframe after reading table `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_feb26_20260301_oop` is:	(1531733, 4)


In [30]:
dt_bs_oop_ex.sample(3)

,ee_customer_id,calc_date,target_dev,score_oop
566373,1310295,2025-04-01,NaN,0.640933
474734,1201428,2025-02-01,NaN,0.488110
1226172,1204372,2025-02-01,NaN,0.655690


In [31]:
dt_bs_oop_new['ee_customer_id'] = dt_bs_oop_new['ee_customer_id'].astype('str')
dt_bs_oop_ex['ee_customer_id'] = dt_bs_oop_ex['ee_customer_id'].astype('str')

In [32]:
dt_bs_oop_ex["calc_date"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce")
dt_bs_oop_ex["calc_date_correct"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_oop_ex['calc_date_ym'] = dt_bs_oop_ex['calc_date_correct'].dt.year*100 + dt_bs_oop_ex['calc_date_correct'].dt.month
dt_bs_oop_ex.sample(3)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym
297718,1008343,2024-08-01,NaN,0.576962,2024-07-01,202407
1527572,1582848,2026-02-01,NaN,0.613718,2026-01-01,202601
834194,1181185,2026-01-01,NaN,0.607835,2025-12-01,202512


# Gini

## prod new

In [33]:
dt_prod_api = dt_prod_api.merge(
    dt_raw_slim[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'score_oop', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

dt_prod_api.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
56786,1452819,2025-08-23T07:41:14.546720,3,3,0.542531,B,NaT,<NA>,NaN,NaN,NaN,NaN
52070,1571574,2025-12-14T07:54:51.577090,4,4,0.668543,B,2025-12-15 10:57:18,0,0.489005,12628.442,12628.442,12628.442
62781,1457006,2025-07-16T06:07:28.279322,4,4,0.501680,C,2025-07-21 12:26:37,<NA>,0.523340,NaN,NaN,17005.941


In [34]:
dt_prod_api['onb_rd_diff'] = (abs(pd.to_datetime(dt_prod_api['run_date']).dt.normalize() - pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize())).dt.days
dt_prod_api['ee_onboarding_date'] = pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize()
dt_prod_api['run_date'] = pd.to_datetime(dt_prod_api['run_date']).dt.normalize()
dt_prod_api['onboarding_date_ym'] = dt_prod_api['ee_onboarding_date'].dt.year*100 + dt_prod_api['ee_onboarding_date'].dt.month
dt_prod_api['run_date_ym'] = dt_prod_api['run_date'].dt.year*100 + dt_prod_api['run_date'].dt.month

dt_prod_api.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
13590,1553999,2026-03-09,6,6,0.433351,D,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202603
142913,1482998,2025-09-16,10-12,12,0.576819,A,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202509
60733,1606375,2026-02-20,3,3,0.508707,C,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202602


In [35]:
dt_prod_api_calc = (dt_prod_api
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

dt_prod_api_calc.sample(3)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
91274,1409198,2025-07-03,4,4,0.605123,A,2025-07-05,0,0.568861,15735.485,15735.485,15735.485,2.0,202507.0,202507
162323,1148226,2025-11-03,7,7,0.453750,D,2024-02-09,0,0.458005,8451.041,8451.041,8451.041,633.0,202402.0,202511
121758,1128273,2026-01-26,5,5,0.304497,E,2023-11-21,0,0.311506,0.000,0.000,5901.767,797.0,202311.0,202601


### OOP New Users Prod

In [36]:
print('OOP New Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUserProd = calculate_gini_for_table(
    data = dt_prod_api_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopNewUserProd

OOP New Users Prod

Gini Coefficient Results:
Jun 2025: 0.25 (Sample size: 15)
Jul 2025: 0.0186 (Sample size: 122)
Aug 2025: 0.0423 (Sample size: 86)
Sep 2025: -0.2492 (Sample size: 108)
Jun 2025 - Sep 2025: -0.0492 (Sample size: 331)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           15 53.333333            0.2500
           Jul 2025 2025-07-01 2025-07-31          122 67.213115            0.0186
           Aug 2025 2025-08-01 2025-08-31           86 69.767442            0.0423
           Sep 2025 2025-09-01 2025-09-30          108 73.148148           -0.2492
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          331 69.184290           -0.0492


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,15,53.333333,0.2500
1,Jul 2025,2025-07-01,2025-07-31,122,67.213115,0.0186
2,Aug 2025,2025-08-01,2025-08-31,86,69.767442,0.0423
3,Sep 2025,2025-09-01,2025-09-30,108,73.148148,-0.2492
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,331,69.184290,-0.0492


### OOP New Users Prod BS

In [37]:
print('OOP New Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdBS = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['score_oop'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopNewUsersProdBS

OOP New Users Prod BS

Gini Coefficient Results:
Jun 2025: 0.0357 (Sample size: 15)
Jul 2025: 0.326 (Sample size: 115)
Aug 2025: 0.0579 (Sample size: 81)
Sep 2025: -0.1983 (Sample size: 103)
Jun 2025 - Sep 2025: 0.0552 (Sample size: 314)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           15 53.333333            0.0357
           Jul 2025 2025-07-01 2025-07-31          115 66.956522            0.3260
           Aug 2025 2025-08-01 2025-08-31           81 69.135802            0.0579
           Sep 2025 2025-09-01 2025-09-30          103 73.786408           -0.1983
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          314 69.108280            0.0552


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,15,53.333333,0.0357
1,Jul 2025,2025-07-01,2025-07-31,115,66.956522,0.3260
2,Aug 2025,2025-08-01,2025-08-31,81,69.135802,0.0579
3,Sep 2025,2025-09-01,2025-09-30,103,73.786408,-0.1983
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,314,69.108280,0.0552


### OOP New Users Prod, weighted as-is

In [38]:
print('OOP New Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdwgasis = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

oopNewUsersProdwgasis

OOP New Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.3099 (Sample size: 14)
Jul 2025: -0.2137 (Sample size: 105)
Aug 2025: 0.098 (Sample size: 76)
Sep 2025: -0.2504 (Sample size: 102)
Jun 2025 - Sep 2025: -0.1222 (Sample size: 297)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.3099
           Jul 2025 2025-07-01 2025-07-31          105 69.523810           -0.2137
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0980
           Sep 2025 2025-09-01 2025-09-30          102 74.509804           -0.2504
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          297 71.043771           -0.1222


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.3099
1,Jul 2025,2025-07-01,2025-07-31,105,69.523810,-0.2137
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0980
3,Sep 2025,2025-09-01,2025-09-30,102,74.509804,-0.2504
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,297,71.043771,-0.1222


In [39]:
dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_api_calc['osbal_as_of_oop_eligible_date'])

In [40]:
dt_prod_api_calc.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym,osbal_as_of_oop_eligible_date_log
27873,1534716,2025-09-02,3,3,0.510708,B,2025-09-02,0,0.536357,6979.460,6979.460,6979.460,0.0,202509.0,202509,8.850870
67868,1205676,2025-10-24,5,5,0.151311,F,2024-06-06,0,0.156823,9397.272,9397.272,9397.272,505.0,202406.0,202510,9.148281
89206,1606237,2025-11-04,4,4,0.229425,F,2025-11-04,1,0.229425,67649.081,67649.081,96368.574,0.0,202511.0,202511,11.122104
50746,1179971,2025-11-10,6,6,0.553731,B,2024-04-09,0,0.563798,16383.617,8125.908,8125.908,580.0,202404.0,202511,9.002936
63284,1569812,2025-10-03,4,4,0.713786,A,2025-10-03,1,0.687167,8521.345,969.749,0.000,0.0,202510.0,202510,6.878068


### OOP New Users Prod, weighted log

In [41]:
print('OOP New Users Prod, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersProdwglog = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdwglog

OOP New Users Prod, weighted log

Gini Coefficient Results:
Jun 2025: 0.212 (Sample size: 14)
Jul 2025: -0.029 (Sample size: 105)
Aug 2025: 0.0833 (Sample size: 76)
Sep 2025: -0.2679 (Sample size: 102)
Oct 2025: 0.0553 (Sample size: 40)
Nov 2025: 0.3789 (Sample size: 60)
Dec 2025: -0.2941 (Sample size: 26)
Jun 2025 - Dec 2025: -0.0167 (Sample size: 423)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.2120
           Jul 2025 2025-07-01 2025-07-31          105 69.523810           -0.0290
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0833
           Sep 2025 2025-09-01 2025-09-30          102 74.509804           -0.2679
           Oct 2025 2025-10-01 2025-10-31           40 67.500000            0.0553
           Nov 2025 2025-11-01 2025-11-30           60 68.333333            0.3789
           Dec 2025 2025-12-01 2025-12-31      

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.2120
1,Jul 2025,2025-07-01,2025-07-31,105,69.523810,-0.0290
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0833
3,Sep 2025,2025-09-01,2025-09-30,102,74.509804,-0.2679
4,Oct 2025,2025-10-01,2025-10-31,40,67.500000,0.0553
5,Nov 2025,2025-11-01,2025-11-30,60,68.333333,0.3789
6,Dec 2025,2025-12-01,2025-12-31,26,50.000000,-0.2941
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,423,69.030733,-0.0167


### OOP New Users Prod, weighted log - 4 months

In [42]:
print('OOP New Users Prod, weighted log - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdwglog4mnt = calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdwglog4mnt

OOP New Users Prod, weighted log - 4 months

Gini Coefficient Results:
Jun 2025: 0.212 (Sample size: 14)
Jul 2025: -0.029 (Sample size: 105)
Aug 2025: 0.0833 (Sample size: 76)
Sep 2025: -0.2679 (Sample size: 102)
Jun 2025 - Sep 2025: -0.0682 (Sample size: 297)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.2120
           Jul 2025 2025-07-01 2025-07-31          105 69.523810           -0.0290
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0833
           Sep 2025 2025-09-01 2025-09-30          102 74.509804           -0.2679
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          297 71.043771           -0.0682


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.2120
1,Jul 2025,2025-07-01,2025-07-31,105,69.523810,-0.0290
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0833
3,Sep 2025,2025-09-01,2025-09-30,102,74.509804,-0.2679
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,297,71.043771,-0.0682


### OOP New Users Prod BS, weighted as-is

In [43]:
print('OOP New Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdBSwgasis = calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

oopNewUsersProdBSwgasis

OOP New Users Prod BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2166 (Sample size: 14)
Jul 2025: 0.0825 (Sample size: 105)
Aug 2025: -0.2214 (Sample size: 76)
Sep 2025: -0.2836 (Sample size: 102)
Jun 2025 - Sep 2025: -0.0597 (Sample size: 297)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.2166
           Jul 2025 2025-07-01 2025-07-31          105 69.523810            0.0825
           Aug 2025 2025-08-01 2025-08-31           76 71.052632           -0.2214
           Sep 2025 2025-09-01 2025-09-30          102 74.509804           -0.2836
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          297 71.043771           -0.0597


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.2166
1,Jul 2025,2025-07-01,2025-07-31,105,69.523810,0.0825
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,-0.2214
3,Sep 2025,2025-09-01,2025-09-30,102,74.509804,-0.2836
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,297,71.043771,-0.0597


### OOP New Users Prod BS, weighted log transformed

In [44]:
print('OOP New Users Prod BS, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersProdBSwglogtranformed = calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdBSwglogtranformed

OOP New Users Prod BS, weighted log transformed

Gini Coefficient Results:
Jun 2025: 0.0937 (Sample size: 14)
Jul 2025: 0.2474 (Sample size: 105)
Aug 2025: 0.0318 (Sample size: 76)
Sep 2025: -0.2318 (Sample size: 102)
Oct 2025: 0.0734 (Sample size: 40)
Nov 2025: 0.3944 (Sample size: 60)
Dec 2025: -0.0829 (Sample size: 26)
Jun 2025 - Dec 2025: 0.0563 (Sample size: 423)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.0937
           Jul 2025 2025-07-01 2025-07-31          105 69.523810            0.2474
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0318
           Sep 2025 2025-09-01 2025-09-30          102 74.509804           -0.2318
           Oct 2025 2025-10-01 2025-10-31           40 67.500000            0.0734
           Nov 2025 2025-11-01 2025-11-30           60 68.333333            0.3944
           Dec 2025 2025-12-01 2

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.0937
1,Jul 2025,2025-07-01,2025-07-31,105,69.523810,0.2474
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0318
3,Sep 2025,2025-09-01,2025-09-30,102,74.509804,-0.2318
4,Oct 2025,2025-10-01,2025-10-31,40,67.500000,0.0734
5,Nov 2025,2025-11-01,2025-11-30,60,68.333333,0.3944
6,Dec 2025,2025-12-01,2025-12-31,26,50.000000,-0.0829
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,423,69.030733,0.0563


### OOP New Users Prod BS, weighted log transformed - 4 months

In [45]:
print('OOP New Users Prod BS, weighted log transformed - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersProdBSwglogtransformed4mnt = calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

oopNewUsersProdBSwglogtransformed4mnt

OOP New Users Prod BS, weighted log transformed - 4 months

Gini Coefficient Results:
Jun 2025: 0.0937 (Sample size: 14)
Jul 2025: 0.2474 (Sample size: 105)
Aug 2025: 0.0318 (Sample size: 76)
Sep 2025: -0.2318 (Sample size: 102)
Jun 2025 - Sep 2025: 0.0094 (Sample size: 297)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30           14 57.142857            0.0937
           Jul 2025 2025-07-01 2025-07-31          105 69.523810            0.2474
           Aug 2025 2025-08-01 2025-08-31           76 71.052632            0.0318
           Sep 2025 2025-09-01 2025-09-30          102 74.509804           -0.2318
Jun 2025 - Sep 2025 2025-06-01 2025-09-30          297 71.043771            0.0094


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,14,57.142857,0.0937
1,Jul 2025,2025-07-01,2025-07-31,105,69.523810,0.2474
2,Aug 2025,2025-08-01,2025-08-31,76,71.052632,0.0318
3,Sep 2025,2025-09-01,2025-09-30,102,74.509804,-0.2318
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,297,71.043771,0.0094


## prod existing

In [46]:
dt_prod_batch.shape

(407568, 6)

In [47]:
dt_prod_batch['ee_customer_id'].nunique()

80348

In [48]:
dt_prod_batch = dt_prod_batch.drop_duplicates(subset=['ee_customer_id','run_date','attrition_time_to_leave','oop_score_prod'])

In [49]:
dt_prod_batch.shape

(406093, 6)

In [50]:
dt_prod_batch["run_date"] = pd.to_datetime(dt_prod_batch["run_date"], errors="coerce")
dt_prod_batch['run_date_ym'] = dt_prod_batch['run_date'].dt.year*100 + dt_prod_batch['run_date'].dt.month

In [51]:
dt_prod_batch.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym
130584,1207001,2026-03-28,Average,9,0.461844,C,202603
233643,1218292,2026-02-17,Average,9,0.572116,B,202602
375033,1151174,2025-08-31,Average,8,0.520669,B,202508
114205,1065642,2026-02-14,Very low,12+,0.520720,C,202602
271906,1217921,2025-08-31,Low,11,0.517900,B,202508


In [52]:
dt_prod_batch = dt_prod_batch.merge(
    dt_raw_slim[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_ex[['ee_customer_id', 'calc_date_ym', 'score_oop']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [53]:
dt_prod_batch.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop
81882,1694104,2026-02-26,Average,8,0.498098,C,202602,2026-01-26 12:33:21,<NA>,NaN,NaN,15249.232,NaN,NaN
316304,1475102,2025-08-05,Very low,12+,0.548588,B,202508,2025-07-05 16:27:47,<NA>,0.0,NaN,0.000,NaN,NaN
93710,1197517,2026-02-28,Average,8,0.603513,B,202602,2024-05-31 12:41:20,<NA>,NaN,NaN,7030.851,NaN,NaN
28470,1546712,2026-03-17,Very low,12+,0.572422,B,202603,2025-09-17 12:42:58,<NA>,NaN,NaN,26907.237,NaN,NaN
137872,1155685,2026-01-10,Average,7,0.530494,C,202601,2024-02-10 16:20:59,<NA>,NaN,NaN,3781.519,202601.0,0.487661


In [54]:
dt_prod_batch.rename(columns={'ee_onboarding_date_x':'ee_onboarding_date', }, inplace=True)

In [55]:
dt_prod_batch['ee_onboarding_date'] = pd.to_datetime(dt_prod_batch['ee_onboarding_date']).dt.normalize()
dt_prod_batch['onboarding_date_ym'] = dt_prod_batch['ee_onboarding_date'].dt.year*100 + dt_prod_batch['ee_onboarding_date'].dt.month
dt_prod_batch['onb_rd_diff'] = (abs(dt_prod_batch['run_date'] - dt_prod_batch['ee_onboarding_date'])).dt.days

In [56]:
dt_prod_batch.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop,onboarding_date_ym,onb_rd_diff
292288,1146925,2025-09-18,Very low,12+,0.549686,B,202509,2024-01-18,<NA>,NaN,NaN,0.000,202509.0,0.704308,202401,609
148889,1465135,2025-10-24,High,5,0.344440,F,202510,2025-06-24,<NA>,NaN,NaN,2984.516,202510.0,0.503338,202506,122
42339,1193545,2025-12-09,High,5,0.575182,B,202512,2024-05-09,<NA>,NaN,NaN,7236.093,202512.0,0.378614,202405,579
401604,1248783,2025-07-08,Average,9,0.550580,B,202507,2024-08-08,<NA>,NaN,NaN,25992.311,202507.0,0.646424,202408,334
307279,1298133,2025-08-10,Very low,12+,0.585466,A,202508,2024-12-10,<NA>,NaN,NaN,13854.917,202508.0,0.668698,202412,243


In [57]:
dt_prod_batch_calc = (dt_prod_batch
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['ee_customer_id','run_date']))

In [58]:
print(f"The shape of dt_prod_batch is:\t{dt_prod_batch.shape}")
print(f"The shape of dt_prod_batch_calc after dropping na is:\t{dt_prod_batch_calc.shape}")

The shape of dt_prod_batch is:	(406093, 16)
The shape of dt_prod_batch_calc after dropping na is:	(17085, 16)


### OOP Existing Users Prod

In [59]:
print('OOP Existing Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProd = calculate_gini_for_table(
    data = dt_prod_batch_calc,
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopExtUsersProd

OOP Existing Users Prod

Gini Coefficient Results:
Jun 2025: 0.2641 (Sample size: 2,153)
Jul 2025: 0.265 (Sample size: 3,489)
Aug 2025: 0.2545 (Sample size: 2,783)
Sep 2025: 0.2574 (Sample size: 2,178)
Jun 2025 - Sep 2025: 0.259 (Sample size: 10,603)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         2138 75.210477            0.2641
           Jul 2025 2025-07-01 2025-07-31         3489 73.029521            0.2650
           Aug 2025 2025-08-01 2025-08-31         2783 70.858785            0.2545
           Sep 2025 2025-09-01 2025-09-30         2178 69.146006            0.2574
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         4448 73.538669            0.2590


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,2138,75.210477,0.2641
1,Jul 2025,2025-07-01,2025-07-31,3489,73.029521,0.2650
2,Aug 2025,2025-08-01,2025-08-31,2783,70.858785,0.2545
3,Sep 2025,2025-09-01,2025-09-30,2178,69.146006,0.2574
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,4448,73.538669,0.2590


### OOP Existing Users Prod BS

In [60]:
print('OOP Existing Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdBS = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['score_oop'].notna()],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopExtUsersProdBS

OOP Existing Users Prod BS

Gini Coefficient Results:
Jun 2025: 0.3002 (Sample size: 1,594)
Jul 2025: 0.2899 (Sample size: 2,476)
Aug 2025: 0.3045 (Sample size: 1,947)
Sep 2025: 0.2997 (Sample size: 1,554)
Jun 2025 - Sep 2025: 0.2978 (Sample size: 7,571)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1581 75.395319            0.3002
           Jul 2025 2025-07-01 2025-07-31         2476 71.849758            0.2899
           Aug 2025 2025-08-01 2025-08-31         1947 68.772470            0.3045
           Sep 2025 2025-09-01 2025-09-30         1554 67.696268            0.2997
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3215 73.157076            0.2978


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1581,75.395319,0.3002
1,Jul 2025,2025-07-01,2025-07-31,2476,71.849758,0.2899
2,Aug 2025,2025-08-01,2025-08-31,1947,68.772470,0.3045
3,Sep 2025,2025-09-01,2025-09-30,1554,67.696268,0.2997
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3215,73.157076,0.2978


### OOP Existing Users Prod, weighted as-is

In [61]:
print('OOP Existing Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtusersProdwgasis = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtusersProdwgasis

OOP Existing Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2716 (Sample size: 1,917)
Jul 2025: 0.2088 (Sample size: 3,089)
Aug 2025: 0.2019 (Sample size: 2,445)
Sep 2025: 0.1966 (Sample size: 1,932)
Jun 2025 - Sep 2025: 0.2135 (Sample size: 9,383)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1906 76.495278            0.2716
           Jul 2025 2025-07-01 2025-07-31         3089 74.166397            0.2088
           Aug 2025 2025-08-01 2025-08-31         2445 72.188139            0.2019
           Sep 2025 2025-09-01 2025-09-30         1932 71.066253            0.1966
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3945 74.828897            0.2135


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1906,76.495278,0.2716
1,Jul 2025,2025-07-01,2025-07-31,3089,74.166397,0.2088
2,Aug 2025,2025-08-01,2025-08-31,2445,72.188139,0.2019
3,Sep 2025,2025-09-01,2025-09-30,1932,71.066253,0.1966
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3945,74.828897,0.2135


In [62]:
dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_batch_calc['osbal_as_of_oop_eligible_date'])

### OOP Existing Users Prod, weighted as-is - jun25 - Dec25

In [63]:
print('OOP Existing Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdwgasisjun25dec25 = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersProdwgasisjun25dec25

OOP Existing Users Prod, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.2716 (Sample size: 1,917)
Jul 2025: 0.2088 (Sample size: 3,089)
Aug 2025: 0.2019 (Sample size: 2,445)
Sep 2025: 0.1966 (Sample size: 1,932)
Oct 2025: 0.1591 (Sample size: 1,600)
Nov 2025: 0.219 (Sample size: 1,369)
Dec 2025: 0.229 (Sample size: 994)
Jun 2025 - Dec 2025: 0.1982 (Sample size: 13,346)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1906 76.495278            0.2716
           Jul 2025 2025-07-01 2025-07-31         3089 74.166397            0.2088
           Aug 2025 2025-08-01 2025-08-31         2445 72.188139            0.2019
           Sep 2025 2025-09-01 2025-09-30         1932 71.066253            0.1966
           Oct 2025 2025-10-01 2025-10-31         1600 69.250000            0.1591
           Nov 2025 2025-11-01 2025-11-30         1369 68.809350            0.2190
           Dec 2025 2025

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1906,76.495278,0.2716
1,Jul 2025,2025-07-01,2025-07-31,3089,74.166397,0.2088
2,Aug 2025,2025-08-01,2025-08-31,2445,72.188139,0.2019
3,Sep 2025,2025-09-01,2025-09-30,1932,71.066253,0.1966
4,Oct 2025,2025-10-01,2025-10-31,1600,69.250000,0.1591
5,Nov 2025,2025-11-01,2025-11-30,1369,68.809350,0.2190
6,Dec 2025,2025-12-01,2025-12-31,994,64.587525,0.2290
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,4400,74.545455,0.1982


### OOP Existing Users Prod, weighted log transformed

In [64]:
print('OOP Existing Users Prod, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdwglogtransformed = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdwglogtransformed

OOP Existing Users Prod, weighted log transformed

Gini Coefficient Results:
Jun 2025: 0.2697 (Sample size: 1,917)
Jul 2025: 0.2638 (Sample size: 3,089)
Aug 2025: 0.2617 (Sample size: 2,445)
Sep 2025: 0.2656 (Sample size: 1,932)
Oct 2025: 0.2534 (Sample size: 1,600)
Nov 2025: 0.2215 (Sample size: 1,369)
Dec 2025: 0.217 (Sample size: 994)
Jun 2025 - Dec 2025: 0.2423 (Sample size: 13,346)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1906 76.495278            0.2697
           Jul 2025 2025-07-01 2025-07-31         3089 74.166397            0.2638
           Aug 2025 2025-08-01 2025-08-31         2445 72.188139            0.2617
           Sep 2025 2025-09-01 2025-09-30         1932 71.066253            0.2656
           Oct 2025 2025-10-01 2025-10-31         1600 69.250000            0.2534
           Nov 2025 2025-11-01 2025-11-30         1369 68.809350            0.2215
           De

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1906,76.495278,0.2697
1,Jul 2025,2025-07-01,2025-07-31,3089,74.166397,0.2638
2,Aug 2025,2025-08-01,2025-08-31,2445,72.188139,0.2617
3,Sep 2025,2025-09-01,2025-09-30,1932,71.066253,0.2656
4,Oct 2025,2025-10-01,2025-10-31,1600,69.250000,0.2534
5,Nov 2025,2025-11-01,2025-11-30,1369,68.809350,0.2215
6,Dec 2025,2025-12-01,2025-12-31,994,64.587525,0.2170
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,4400,74.545455,0.2423


### OOP Existing Users Prod, weighted log transformed - 4 months

In [65]:
print('OOP Existing Users Prod, weighted log transformed - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdwglogtransformed4mnt = calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdwglogtransformed4mnt

OOP Existing Users Prod, weighted log transformed - 4 months

Gini Coefficient Results:
Jun 2025: 0.2697 (Sample size: 1,917)
Jul 2025: 0.2638 (Sample size: 3,089)
Aug 2025: 0.2617 (Sample size: 2,445)
Sep 2025: 0.2656 (Sample size: 1,932)
Jun 2025 - Sep 2025: 0.2637 (Sample size: 9,383)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1906 76.495278            0.2697
           Jul 2025 2025-07-01 2025-07-31         3089 74.166397            0.2638
           Aug 2025 2025-08-01 2025-08-31         2445 72.188139            0.2617
           Sep 2025 2025-09-01 2025-09-30         1932 71.066253            0.2656
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         3945 74.828897            0.2637


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1906,76.495278,0.2697
1,Jul 2025,2025-07-01,2025-07-31,3089,74.166397,0.2638
2,Aug 2025,2025-08-01,2025-08-31,2445,72.188139,0.2617
3,Sep 2025,2025-09-01,2025-09-30,1932,71.066253,0.2656
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,3945,74.828897,0.2637


### OOP Existing Users Prod BS, weighted as-is

In [66]:
print('OOP Existing Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdBSwgasis = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersProdBSwgasis

OOP Existing Users Prod BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.3192 (Sample size: 1,401)
Jul 2025: 0.2131 (Sample size: 2,165)
Aug 2025: 0.2387 (Sample size: 1,685)
Sep 2025: 0.2434 (Sample size: 1,377)
Jun 2025 - Sep 2025: 0.2455 (Sample size: 6,628)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1391 76.779295            0.3192
           Jul 2025 2025-07-01 2025-07-31         2165 73.256351            0.2131
           Aug 2025 2025-08-01 2025-08-31         1685 70.267062            0.2387
           Sep 2025 2025-09-01 2025-09-30         1377 69.208424            0.2434
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         2832 74.540960            0.2455


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1391,76.779295,0.3192
1,Jul 2025,2025-07-01,2025-07-31,2165,73.256351,0.2131
2,Aug 2025,2025-08-01,2025-08-31,1685,70.267062,0.2387
3,Sep 2025,2025-09-01,2025-09-30,1377,69.208424,0.2434
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,2832,74.540960,0.2455


### OOP Existing Users Prod BSm weighted as-is - Jun25-Dec25

In [67]:
print('OOP Existing Users Prod BSm weighted as-is - Jun25-Dec25')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdBSwgasisjun25dec25 = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersProdBSwgasisjun25dec25

OOP Existing Users Prod BSm weighted as-is - Jun25-Dec25

Gini Coefficient Results:
Jun 2025: 0.3192 (Sample size: 1,401)
Jul 2025: 0.2131 (Sample size: 2,165)
Aug 2025: 0.2387 (Sample size: 1,685)
Sep 2025: 0.2434 (Sample size: 1,377)
Oct 2025: 0.2938 (Sample size: 1,156)
Nov 2025: 0.2572 (Sample size: 1,055)
Dec 2025: 0.2224 (Sample size: 644)
Jun 2025 - Dec 2025: 0.2533 (Sample size: 9,483)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1391 76.779295            0.3192
           Jul 2025 2025-07-01 2025-07-31         2165 73.256351            0.2131
           Aug 2025 2025-08-01 2025-08-31         1685 70.267062            0.2387
           Sep 2025 2025-09-01 2025-09-30         1377 69.208424            0.2434
           Oct 2025 2025-10-01 2025-10-31         1156 68.252595            0.2938
           Nov 2025 2025-11-01 2025-11-30         1055 68.436019            0.2572
      

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1391,76.779295,0.3192
1,Jul 2025,2025-07-01,2025-07-31,2165,73.256351,0.2131
2,Aug 2025,2025-08-01,2025-08-31,1685,70.267062,0.2387
3,Sep 2025,2025-09-01,2025-09-30,1377,69.208424,0.2434
4,Oct 2025,2025-10-01,2025-10-31,1156,68.252595,0.2938
5,Nov 2025,2025-11-01,2025-11-30,1055,68.436019,0.2572
6,Dec 2025,2025-12-01,2025-12-31,644,58.850932,0.2224
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3144,74.459288,0.2533


### OOP Existing Users Prod BS, weighted log transformed

In [68]:
print('OOP Existing Users Prod BS, weighted log transformed - 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersProdBSwglogtransformed = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdBSwglogtransformed

OOP Existing Users Prod BS, weighted log transformed - 4 months

Gini Coefficient Results:
Jun 2025: 0.3157 (Sample size: 1,401)
Jul 2025: 0.286 (Sample size: 2,165)
Aug 2025: 0.3083 (Sample size: 1,685)
Sep 2025: 0.2972 (Sample size: 1,377)
Jun 2025 - Sep 2025: 0.3001 (Sample size: 6,628)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1391 76.779295            0.3157
           Jul 2025 2025-07-01 2025-07-31         2165 73.256351            0.2860
           Aug 2025 2025-08-01 2025-08-31         1685 70.267062            0.3083
           Sep 2025 2025-09-01 2025-09-30         1377 69.208424            0.2972
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         2832 74.540960            0.3001


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1391,76.779295,0.3157
1,Jul 2025,2025-07-01,2025-07-31,2165,73.256351,0.2860
2,Aug 2025,2025-08-01,2025-08-31,1685,70.267062,0.3083
3,Sep 2025,2025-09-01,2025-09-30,1377,69.208424,0.2972
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,2832,74.540960,0.3001


### OOP Existing Users Prod BSm weighted log transformed

In [69]:
print('OOP Existing Users Prod BSm weighted log transformed jun25-dec25')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersProdBSwglogtransformed = calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersProdBSwglogtransformed

OOP Existing Users Prod BSm weighted log transformed jun25-dec25

Gini Coefficient Results:
Jun 2025: 0.3157 (Sample size: 1,401)
Jul 2025: 0.286 (Sample size: 2,165)
Aug 2025: 0.3083 (Sample size: 1,685)
Sep 2025: 0.2972 (Sample size: 1,377)
Oct 2025: 0.3273 (Sample size: 1,156)
Nov 2025: 0.3085 (Sample size: 1,055)
Dec 2025: 0.328 (Sample size: 644)
Jun 2025 - Dec 2025: 0.3078 (Sample size: 9,483)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         1391 76.779295            0.3157
           Jul 2025 2025-07-01 2025-07-31         2165 73.256351            0.2860
           Aug 2025 2025-08-01 2025-08-31         1685 70.267062            0.3083
           Sep 2025 2025-09-01 2025-09-30         1377 69.208424            0.2972
           Oct 2025 2025-10-01 2025-10-31         1156 68.252595            0.3273
           Nov 2025 2025-11-01 2025-11-30         1055 68.436019            0.3085


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,1391,76.779295,0.3157
1,Jul 2025,2025-07-01,2025-07-31,2165,73.256351,0.2860
2,Aug 2025,2025-08-01,2025-08-31,1685,70.267062,0.3083
3,Sep 2025,2025-09-01,2025-09-30,1377,69.208424,0.2972
4,Oct 2025,2025-10-01,2025-10-31,1156,68.252595,0.3273
5,Nov 2025,2025-11-01,2025-11-30,1055,68.436019,0.3085
6,Dec 2025,2025-12-01,2025-12-31,644,58.850932,0.3280
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3144,74.459288,0.3078


## bs new

In [70]:
dt_bs_oop_new.shape

(169297, 7)

In [71]:
dt_bs_oop_new['ee_customer_id'].nunique()

169297

In [72]:
dt_bs_oop_new = dt_bs_oop_new.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
)

dt_bs_oop_new.sample(5)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,ee_onboarding_date,oop_target
137854,1393984,2025-05-01,NaN,0.591467,NaN,NaN,8434.593,2025-04-12 11:15:44,<NA>
143227,1546180,2025-10-01,NaN,0.600377,NaN,NaN,0.000,2025-09-14 14:51:18,<NA>
123023,1573021,2025-11-01,NaN,0.570210,NaN,NaN,33742.224,2025-10-07 16:32:18,<NA>
93547,1365455,2026-01-01,NaN,0.524316,NaN,NaN,13348.741,2025-12-04 18:41:33,<NA>
6760,1267215,2024-10-01,0.0,0.432589,6919.516,3622.358,3622.358,2024-09-11 10:46:27,0


In [73]:
dt_bs_oop_new['ee_onboarding_date'] = pd.to_datetime(dt_bs_oop_new['ee_onboarding_date']).dt.normalize()
dt_bs_oop_new['onboarding_date_ym'] = dt_bs_oop_new['ee_onboarding_date'].dt.year*100 + dt_bs_oop_new['ee_onboarding_date'].dt.month

In [74]:
dt_bs_oop_new.sample(5)

,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,ee_onboarding_date,oop_target,onboarding_date_ym
126469,1398735,2025-05-01,NaN,0.575582,NaN,NaN,1041.966,2025-04-24,<NA>,202504
21617,1061621,2023-07-01,0.0,0.538444,0.0,9669.914,9669.914,2023-06-11,0,202306
61254,1216143,2024-07-01,NaN,0.445693,NaN,NaN,14672.387,2024-06-14,<NA>,202406
57312,1594231,2025-11-01,NaN,0.429019,NaN,NaN,2454.294,2025-10-23,<NA>,202510
128409,1512671,2026-02-01,NaN,0.578551,NaN,NaN,14463.857,2026-01-03,<NA>,202601


In [75]:
dt_bs_oop_new_calc = dt_bs_oop_new.dropna(subset=['oop_target'])

print(f"The shape of the dataframe dt_bs_oop_new is:\t {dt_bs_oop_new.shape} ")
print(f"The shape of the dataframe after dropping na from oop_target is:\t {dt_bs_oop_new_calc.shape}")

The shape of the dataframe dt_bs_oop_new is:	 (169297, 10) 
The shape of the dataframe after dropping na from oop_target is:	 (31630, 10)


### OOP New Users BS

In [76]:
print('OOP New Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersBS = calculate_gini_for_table(
    data = dt_bs_oop_new_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

oopNewUsersBS

OOP New Users BS

Gini Coefficient Results:
Jun 2025: 0.0792 (Sample size: 475)
Jul 2025: 0.0811 (Sample size: 460)
Aug 2025: 0.0467 (Sample size: 304)
Sep 2025: -0.0115 (Sample size: 646)
Jun 2025 - Sep 2025: 0.0434 (Sample size: 1,885)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          475 69.263158            0.0792
           Jul 2025 2025-07-01 2025-07-31          460 64.565217            0.0811
           Aug 2025 2025-08-01 2025-08-31          304 65.460526            0.0467
           Sep 2025 2025-09-01 2025-09-30          646 69.040248           -0.0115
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1885 67.427056            0.0434


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,475,69.263158,0.0792
1,Jul 2025,2025-07-01,2025-07-31,460,64.565217,0.0811
2,Aug 2025,2025-08-01,2025-08-31,304,65.460526,0.0467
3,Sep 2025,2025-09-01,2025-09-30,646,69.040248,-0.0115
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1885,67.427056,0.0434


In [77]:
dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'])

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_41656\3448610563.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'])


### OOP New Users BS, weighted as-is 4 months

In [78]:
print('OOP New Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersBSwgasis4mnt = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopNewUsersBSwgasis4mnt

OOP New Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: -0.0086 (Sample size: 451)
Jul 2025: 0.0657 (Sample size: 417)
Aug 2025: 0.0903 (Sample size: 282)
Sep 2025: -0.0359 (Sample size: 622)
Jun 2025 - Sep 2025: 0.0114 (Sample size: 1,772)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          451 70.288248           -0.0086
           Jul 2025 2025-07-01 2025-07-31          417 66.906475            0.0657
           Aug 2025 2025-08-01 2025-08-31          282 68.439716            0.0903
           Sep 2025 2025-09-01 2025-09-30          622 70.418006           -0.0359
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1772 69.243792            0.0114


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,451,70.288248,-0.0086
1,Jul 2025,2025-07-01,2025-07-31,417,66.906475,0.0657
2,Aug 2025,2025-08-01,2025-08-31,282,68.439716,0.0903
3,Sep 2025,2025-09-01,2025-09-30,622,70.418006,-0.0359
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1772,69.243792,0.0114


### OOP New Users BS, weighted log - 4 months

In [79]:
print('OOP New Users BS, weighted log 4 months')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopNewUsersBSwglog4mnt = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopNewUsersBSwglog4mnt

OOP New Users BS, weighted log 4 months

Gini Coefficient Results:
Jun 2025: 0.0616 (Sample size: 451)
Jul 2025: 0.0828 (Sample size: 417)
Aug 2025: 0.0458 (Sample size: 282)
Sep 2025: -0.0324 (Sample size: 622)
Jun 2025 - Sep 2025: 0.032 (Sample size: 1,772)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          451 70.288248            0.0616
           Jul 2025 2025-07-01 2025-07-31          417 66.906475            0.0828
           Aug 2025 2025-08-01 2025-08-31          282 68.439716            0.0458
           Sep 2025 2025-09-01 2025-09-30          622 70.418006           -0.0324
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         1772 69.243792            0.0320


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,451,70.288248,0.0616
1,Jul 2025,2025-07-01,2025-07-31,417,66.906475,0.0828
2,Aug 2025,2025-08-01,2025-08-31,282,68.439716,0.0458
3,Sep 2025,2025-09-01,2025-09-30,622,70.418006,-0.0324
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,1772,69.243792,0.0320


### OOP New Users BS, weighted as-is

In [80]:
print('OOP New Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersBSwgasis = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopNewUsersBSwgasis

OOP New Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: -0.0086 (Sample size: 451)
Jul 2025: 0.0657 (Sample size: 417)
Aug 2025: 0.0903 (Sample size: 282)
Sep 2025: -0.0359 (Sample size: 622)
Oct 2025: 0.2523 (Sample size: 951)
Nov 2025: 0.2388 (Sample size: 457)
Dec 2025: 0.1197 (Sample size: 129)
Jun 2025 - Dec 2025: 0.1395 (Sample size: 3,309)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          451 70.288248           -0.0086
           Jul 2025 2025-07-01 2025-07-31          417 66.906475            0.0657
           Aug 2025 2025-08-01 2025-08-31          282 68.439716            0.0903
           Sep 2025 2025-09-01 2025-09-30          622 70.418006           -0.0359
           Oct 2025 2025-10-01 2025-10-31          951 77.287066            0.2523
           Nov 2025 2025-11-01 2025-11-30          457 73.085339            0.2388
           Dec 2025 2025-12-01 2025-12-3

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,451,70.288248,-0.0086
1,Jul 2025,2025-07-01,2025-07-31,417,66.906475,0.0657
2,Aug 2025,2025-08-01,2025-08-31,282,68.439716,0.0903
3,Sep 2025,2025-09-01,2025-09-30,622,70.418006,-0.0359
4,Oct 2025,2025-10-01,2025-10-31,951,77.287066,0.2523
5,Nov 2025,2025-11-01,2025-11-30,457,73.085339,0.2388
6,Dec 2025,2025-12-01,2025-12-31,129,54.263566,0.1197
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3309,71.501964,0.1395


### OOP New Users BS, weighted log

In [81]:
print('OOP New Users BS, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopNewUsersBSwglog = calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopNewUsersBSwglog

OOP New Users BS, weighted log

Gini Coefficient Results:
Jun 2025: 0.0616 (Sample size: 451)
Jul 2025: 0.0828 (Sample size: 417)
Aug 2025: 0.0458 (Sample size: 282)
Sep 2025: -0.0324 (Sample size: 622)
Oct 2025: 0.2755 (Sample size: 951)
Nov 2025: 0.2862 (Sample size: 457)
Dec 2025: 0.1117 (Sample size: 129)
Jun 2025 - Dec 2025: 0.146 (Sample size: 3,309)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30          451 70.288248            0.0616
           Jul 2025 2025-07-01 2025-07-31          417 66.906475            0.0828
           Aug 2025 2025-08-01 2025-08-31          282 68.439716            0.0458
           Sep 2025 2025-09-01 2025-09-30          622 70.418006           -0.0324
           Oct 2025 2025-10-01 2025-10-31          951 77.287066            0.2755
           Nov 2025 2025-11-01 2025-11-30          457 73.085339            0.2862
           Dec 2025 2025-12-01 2025-12-31   

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,451,70.288248,0.0616
1,Jul 2025,2025-07-01,2025-07-31,417,66.906475,0.0828
2,Aug 2025,2025-08-01,2025-08-31,282,68.439716,0.0458
3,Sep 2025,2025-09-01,2025-09-30,622,70.418006,-0.0324
4,Oct 2025,2025-10-01,2025-10-31,951,77.287066,0.2755
5,Nov 2025,2025-11-01,2025-11-30,457,73.085339,0.2862
6,Dec 2025,2025-12-01,2025-12-31,129,54.263566,0.1117
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,3309,71.501964,0.1460


## bs existing

In [82]:
dt_bs_oop_ex.shape

(1531733, 6)

In [83]:
dt_bs_oop_ex['ee_customer_id'].nunique()

144497

In [84]:
dt_bs_oop_ex = dt_bs_oop_ex.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

dt_bs_oop_ex.sample(5)

,ee_customer_id,calc_date,target_dev,score_oop,calc_date_correct,calc_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
263901,1186781,2024-11-01,NaN,0.522103,2024-10-01,202410,2024-07-17 14:02:52,<NA>,NaN,NaN,18278.846
409218,1208796,2025-04-01,NaN,0.452171,2025-03-01,202503,2024-06-03 16:29:48,<NA>,NaN,NaN,13215.902
415785,1235286,2025-04-01,NaN,0.428391,2025-03-01,202503,2024-07-19 14:04:01,<NA>,NaN,NaN,15564.091
454541,1248165,2025-01-01,NaN,0.446112,2024-12-01,202412,2024-09-17 13:19:37,<NA>,NaN,NaN,4353.585
1053494,1390292,2026-02-01,NaN,0.453553,2026-01-01,202601,2025-06-17 11:15:22,<NA>,NaN,NaN,8150.935


In [85]:
dt_bs_oop_ex.shape

(1531733, 11)

In [86]:
dt_bs_oop_ex_calc = dt_bs_oop_ex.dropna(subset=['oop_target'])
dt_bs_oop_ex_calc.shape

(193079, 11)

### OOP Existing Users BS

In [87]:
print('OOP Existing Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtusersBS = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc,
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)
oopExtusersBS

OOP Existing Users BS

Gini Coefficient Results:
Jun 2025: 0.2783 (Sample size: 4,851)
Jul 2025: 0.2674 (Sample size: 5,513)
Aug 2025: 0.2808 (Sample size: 4,793)
Sep 2025: 0.2725 (Sample size: 3,960)
Jun 2025 - Sep 2025: 0.2804 (Sample size: 19,117)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         4851 77.571635            0.2783
           Jul 2025 2025-07-01 2025-07-31         5513 73.952476            0.2674
           Aug 2025 2025-08-01 2025-08-31         4793 71.959107            0.2808
           Sep 2025 2025-09-01 2025-09-30         3960 69.772727            0.2725
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         7226 74.467202            0.2804


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,4851,77.571635,0.2783
1,Jul 2025,2025-07-01,2025-07-31,5513,73.952476,0.2674
2,Aug 2025,2025-08-01,2025-08-31,4793,71.959107,0.2808
3,Sep 2025,2025-09-01,2025-09-30,3960,69.772727,0.2725
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,7226,74.467202,0.2804


In [88]:
dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'])

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_41656\3634989807.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'])


### OOP Existing Users BS, weighted as-is 4 months

In [89]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersBSwgasis4mnt = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersBSwgasis4mnt

OOP Existing Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.264 (Sample size: 3,959)
Jul 2025: 0.2684 (Sample size: 4,660)
Aug 2025: 0.2981 (Sample size: 4,053)
Sep 2025: 0.2745 (Sample size: 3,417)
Jun 2025 - Sep 2025: 0.2833 (Sample size: 16,089)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3959 78.125789            0.2640
           Jul 2025 2025-07-01 2025-07-31         4660 74.420601            0.2684
           Aug 2025 2025-08-01 2025-08-31         4053 72.662226            0.2981
           Sep 2025 2025-09-01 2025-09-30         3417 70.558970            0.2745
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         6217 74.875342            0.2833


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3959,78.125789,0.2640
1,Jul 2025,2025-07-01,2025-07-31,4660,74.420601,0.2684
2,Aug 2025,2025-08-01,2025-08-31,4053,72.662226,0.2981
3,Sep 2025,2025-09-01,2025-09-30,3417,70.558970,0.2745
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,6217,74.875342,0.2833


### OOP Existing Users BS, weighted log transform 4 months

In [90]:
print('OOP Existing Users BS, weighted log transform')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

oopExtUsersBSwglog4mnt = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersBSwglog4mnt

OOP Existing Users BS, weighted log transform

Gini Coefficient Results:
Jun 2025: 0.2878 (Sample size: 3,959)
Jul 2025: 0.2706 (Sample size: 4,660)
Aug 2025: 0.288 (Sample size: 4,053)
Sep 2025: 0.2696 (Sample size: 3,417)
Jun 2025 - Sep 2025: 0.2857 (Sample size: 16,089)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3959 78.125789            0.2878
           Jul 2025 2025-07-01 2025-07-31         4660 74.420601            0.2706
           Aug 2025 2025-08-01 2025-08-31         4053 72.662226            0.2880
           Sep 2025 2025-09-01 2025-09-30         3417 70.558970            0.2696
Jun 2025 - Sep 2025 2025-06-01 2025-09-30         6217 74.875342            0.2857


,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3959,78.125789,0.2878
1,Jul 2025,2025-07-01,2025-07-31,4660,74.420601,0.2706
2,Aug 2025,2025-08-01,2025-08-31,4053,72.662226,0.2880
3,Sep 2025,2025-09-01,2025-09-30,3417,70.558970,0.2696
4,Jun 2025 - Sep 2025,2025-06-01,2025-09-30,6217,74.875342,0.2857


### OOP Existing Users BS, weighted as-is jun25-dec25

In [91]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersBSwgasisjun25dec25 = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

oopExtUsersBSwgasisjun25dec25

OOP Existing Users BS, weighted as-is

Gini Coefficient Results:
Jun 2025: 0.264 (Sample size: 3,959)
Jul 2025: 0.2684 (Sample size: 4,660)
Aug 2025: 0.2981 (Sample size: 4,053)
Sep 2025: 0.2745 (Sample size: 3,417)
Oct 2025: 0.2459 (Sample size: 2,989)
Nov 2025: 0.2672 (Sample size: 2,657)
Dec 2025: 0.2732 (Sample size: 1,934)
Jun 2025 - Dec 2025: 0.2759 (Sample size: 23,669)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3959 78.125789            0.2640
           Jul 2025 2025-07-01 2025-07-31         4660 74.420601            0.2684
           Aug 2025 2025-08-01 2025-08-31         4053 72.662226            0.2981
           Sep 2025 2025-09-01 2025-09-30         3417 70.558970            0.2745
           Oct 2025 2025-10-01 2025-10-31         2989 68.384075            0.2459
           Nov 2025 2025-11-01 2025-11-30         2657 67.406850            0.2672
           Dec 2025 202

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3959,78.125789,0.2640
1,Jul 2025,2025-07-01,2025-07-31,4660,74.420601,0.2684
2,Aug 2025,2025-08-01,2025-08-31,4053,72.662226,0.2981
3,Sep 2025,2025-09-01,2025-09-30,3417,70.558970,0.2745
4,Oct 2025,2025-10-01,2025-10-31,2989,68.384075,0.2459
5,Nov 2025,2025-11-01,2025-11-30,2657,67.406850,0.2672
6,Dec 2025,2025-12-01,2025-12-31,1934,61.633919,0.2732
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,6922,73.692574,0.2759


### OOP Existing Users BS, weighted log transform jun25-dec25

In [92]:
print('OOP Existing Users BS, weighted log transform')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

oopExtUsersBSwglogjun25dec25 = calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

oopExtUsersBSwglogjun25dec25

OOP Existing Users BS, weighted log transform

Gini Coefficient Results:
Jun 2025: 0.2878 (Sample size: 3,959)
Jul 2025: 0.2706 (Sample size: 4,660)
Aug 2025: 0.288 (Sample size: 4,053)
Sep 2025: 0.2696 (Sample size: 3,417)
Oct 2025: 0.2555 (Sample size: 2,989)
Nov 2025: 0.2841 (Sample size: 2,657)
Dec 2025: 0.3127 (Sample size: 1,934)
Jun 2025 - Dec 2025: 0.2857 (Sample size: 23,669)

Summary Table:
             Period Start_Date   End_Date  Sample_Size  Bad Rate  Gini_Coefficient
           Jun 2025 2025-06-01 2025-06-30         3959 78.125789            0.2878
           Jul 2025 2025-07-01 2025-07-31         4660 74.420601            0.2706
           Aug 2025 2025-08-01 2025-08-31         4053 72.662226            0.2880
           Sep 2025 2025-09-01 2025-09-30         3417 70.558970            0.2696
           Oct 2025 2025-10-01 2025-10-31         2989 68.384075            0.2555
           Nov 2025 2025-11-01 2025-11-30         2657 67.406850            0.2841
           Dec 

,Period,Start_Date,End_Date,Sample_Size,Bad Rate,Gini_Coefficient
0,Jun 2025,2025-06-01,2025-06-30,3959,78.125789,0.2878
1,Jul 2025,2025-07-01,2025-07-31,4660,74.420601,0.2706
2,Aug 2025,2025-08-01,2025-08-31,4053,72.662226,0.2880
3,Sep 2025,2025-09-01,2025-09-30,3417,70.558970,0.2696
4,Oct 2025,2025-10-01,2025-10-31,2989,68.384075,0.2555
5,Nov 2025,2025-11-01,2025-11-30,2657,67.406850,0.2841
6,Dec 2025,2025-12-01,2025-12-31,1934,61.633919,0.3127
7,Jun 2025 - Dec 2025,2025-06-01,2025-12-31,6922,73.692574,0.2857


# Creating Slide 2

In [93]:
# Calculate averages from the assigned results
avg_gini_new = oopNewUsersProdBS[oopNewUsersProdBS['Period'].isin(['Jul 2025', 'Aug 2025'])]['Gini_Coefficient'].mean()
avg_gini_ex = oopExtUsersBSwgasis4mnt[oopExtUsersBSwgasis4mnt['Period'].isin(['Jul 2025', 'Aug 2025'])]['Gini_Coefficient'].mean()

summary_data = {
    'UserType': ['New users (< 3MOB)', 'Existing users (>= 3 MOB)'],
    'OOP_Repay_Dev_Gini': [0.30, 0.32],
    'OOP_Repay_Prod_Gini_Avg': [round(avg_gini_new, 2), round(avg_gini_ex, 2)],
    'Attrition_Dev_C_Index': [0.66, 0.64],
    'Attrition_Prod_C_Index': [0.60, 0.59]
}

df_summary = pd.DataFrame(summary_data)
df_summary['run_date'] = CURRENT_DATE
df_summary['monthdisplayed'] = 'Jul25-Aug25'
display(df_summary)


,UserType,OOP_Repay_Dev_Gini,OOP_Repay_Prod_Gini_Avg,Attrition_Dev_C_Index,Attrition_Prod_C_Index,run_date,monthdisplayed
0,New users (< 3MOB),0.30,0.19,0.66,0.60,20260331,Jul25-Aug25
1,Existing users (>= 3 MOB),0.32,0.28,0.64,0.59,20260331,Jul25-Aug25


In [94]:
# Upload to BigQuery
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.tendomodelmonitoringdashboardslide1"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(df_summary, table_id, job_config=job_config)
job.result()  # Wait for the job to complete

LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=7e0a3396-ee17-4cb2-941e-214339358adb>

## Distribution metrics

### prod new

In [95]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_api.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_new = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_new = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_new,
    "min_inclusive": edges_new[:-1],
    "max_exclusive": edges_new[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [96]:
dt_prod_api_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_api_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

#### New prod users, prod score, June-Aug

## Slide 3

### slide 3 upper part

In [97]:
# slide 3 upper part
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt
newprodusersprodscorejuneaug25 = pt.reset_index()
newprodusersprodscorejuneaug25

oop_risk_segment_prod oop_target_bad           osbal_as_of_resignation_date  \
                                 count      mean                          sum   
0                     A             82  0.621951                  1068231.475   
1                     B             66   0.69697                   676618.575   
2                     C             43   0.72093                   392620.044   
3                     D              7       1.0                    59443.298   
4                     E              9  0.555556                    48530.317   
5                     F             16     0.625                   179410.149   

  osbal_current_bad php_weighted_outstanding_bad_rate  
                sum                             ratio  
0        793074.907                          0.742419  
1        471075.402                           0.69622  
2        290468.873                          0.739822  
3         44854.565                          0.754577  
4         39893.464                          0.822032  
5        102230.745                          0.569816

In [98]:
newprodusersprodscorejuneaug25.columns

newprodusersprodscorejuneaug25.columns = newprodusersprodscorejuneaug25.columns.droplevel(1)
newprodusersprodscorejuneaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
newprodusersprodscorejuneaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,82,0.621951,1068231.475,793074.907,0.742419
1,B,66,0.69697,676618.575,471075.402,0.69622
2,C,43,0.72093,392620.044,290468.873,0.739822
3,D,7,1.0,59443.298,44854.565,0.754577
4,E,9,0.555556,48530.317,39893.464,0.822032
5,F,16,0.625,179410.149,102230.745,0.569816


### Slide 3 lower part

In [99]:
# Slide 3 lower part
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
newprodusersbsscorejuneaug25 = pt.sort_index(ascending=False).reset_index()
newprodusersbsscorejuneaug25.columns = newprodusersbsscorejuneaug25.columns.droplevel(1)
newprodusersbsscorejuneaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
newprodusersbsscorejuneaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,6,0.833333,92872.273,71395.594,0.76875
1,B,73,0.561644,840196.517,605036.022,0.720113
2,C,70,0.685714,858157.414,604477.696,0.70439
3,D,30,0.7,409444.462,273617.87,0.668266
4,E,18,0.666667,141930.797,104818.379,0.738518
5,F,14,1.0,82252.395,82252.395,1.0


### bs new

In [100]:
dt_bs_oop_new_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_new_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_41656\2931598673.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_new_calc["oop_risk_segment_bs"] = pd.cut(


In [101]:
# New BS users, BS score, June-Aug
df = dt_bs_oop_new_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
newbsusersbsscorejuneaug25 = pt.sort_index(ascending=False).reset_index()
newbsusersbsscorejuneaug25.columns = newbsusersbsscorejuneaug25.columns.droplevel(1)
newbsusersbsscorejuneaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
newbsusersbsscorejuneaug25


,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,26,0.576923,422222.996,257628.073,0.610171
1,B,664,0.641566,8149837.086,4693646.947,0.575919
2,C,305,0.701639,4086128.227,2731164.676,0.668399
3,D,138,0.630435,1918286.121,1027204.63,0.53548
4,E,73,0.726027,1172740.898,676630.721,0.576965
5,F,33,0.909091,369621.079,280475.128,0.758818


### prod existing

In [102]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_batch.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_ex = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_ex = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_ex,
    "min_inclusive": edges_ex[:-1],
    "max_exclusive": edges_ex[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [103]:
dt_prod_batch_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_batch_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [104]:
dt_prod_batch_calc.to_csv(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\slide4upperpartdata.csv", index = False)

In [105]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)
extprodusersprodscorejune = pt.sort_index(ascending=True).reset_index()

extprodusersprodscorejune.columns = extprodusersprodscorejune.columns.droplevel(1)
extprodusersprodscorejune.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersprodscorejune


,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,78,0.602564,1.508865e+06,737121.067,0.488527
1,B,481,0.648649,8.490016e+06,5099753.503,0.600677
2,C,684,0.719298,1.076251e+07,6776715.727,0.629659
3,D,519,0.842004,7.666802e+06,6085088.997,0.793693
4,E,206,0.820388,2.659963e+06,2096747.019,0.788262
5,F,185,0.864865,2.178477e+06,1845814.173,0.847296


## Slide 4

### Slide 4 upper part

In [223]:
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()
df = df.drop_duplicates(keep='first')

df["oop_target_bad"] = 1 - df["oop_target"]
df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

# ✅ Vectorized filtering: only keep ID when oom_target_bad = 1
df["ee_customer_id_bad"] = df["ee_customer_id"].where(df["oop_target_bad"] == 1, None)

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=[
        "ee_customer_id_bad",
        "oop_target_bad",
        "osbal_as_of_resignation_date",
        "osbal_current_bad",
    ],
    aggfunc={
        "ee_customer_id_bad": pd.Series.nunique,   # ✅ unique bad IDs only
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum",
    }
)

# PHP-weighted O/S bad rate
pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] /
    pt[("osbal_as_of_resignation_date", "sum")]
)

pt = pt.sort_index(ascending=True)
pt

ee_customer_id_bad oop_target_bad            \
                                 nunique          count      mean   
oop_risk_segment_prod                                               
A                                     63            111  0.567568   
B                                    491            784  0.626276   
C                                    741           1066  0.695122   
D                                    697            871   0.80023   
E                                    298            360  0.827778   
F                                    258            297  0.868687   

                      osbal_as_of_resignation_date osbal_current_bad  \
                                               sum               sum   
oop_risk_segment_prod                                                  
A                                     3.846314e+06       1091715.512   
B                                     2.330524e+07       9170897.279   
C                                     3.179328e+07      11443088.254   
D                                     2.184185e+07       10582072.03   
E                                     8.006820e+06       4056979.298   
F                                     5.096132e+06       2853484.516   

                      php_weighted_outstanding_bad_rate  
                                                  ratio  
oop_risk_segment_prod                                    
A                                              0.283834  
B                                              0.393512  
C                                              0.359922  
D                                              0.484486  
E                                               0.50669  
F                                              0.559931

In [106]:
# Slide 4 upper part
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()
df = df.drop_duplicates(keep='first')
df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)
extproduserprodscorejuly = pt.sort_index(ascending=True).reset_index()
extproduserprodscorejuly.columns = extproduserprodscorejuly.columns.droplevel(1)
extproduserprodscorejuly.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extproduserprodscorejuly

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,111,0.567568,2.088233e+06,1091715.512,0.522794
1,B,784,0.626276,1.410375e+07,9170897.279,0.650245
2,C,1066,0.695122,1.865014e+07,11443088.254,0.613566
3,D,871,0.80023,1.381477e+07,10582072.03,0.765997
4,E,360,0.827778,5.177063e+06,4056979.298,0.783645
5,F,297,0.868687,3.426949e+06,2853484.516,0.83266


In [107]:
# Existing prod users, prod score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)
extprodusersprodscoreaug = pt.sort_index(ascending=True).reset_index()
extprodusersprodscoreaug.columns = extprodusersprodscoreaug.columns.droplevel(1)
extprodusersprodscoreaug.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersprodscoreaug

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,91,0.582418,1.655044e+06,900558.606,0.54413
1,B,641,0.613105,1.071291e+07,6858442.901,0.640204
2,C,824,0.650485,1.416886e+07,8624030.107,0.608661
3,D,709,0.788434,1.098754e+07,8373502.95,0.762091
4,E,287,0.818815,3.876952e+06,3121945.393,0.805258
5,F,231,0.848485,2.482998e+06,2137092.277,0.86069


In [108]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extprodusersbsscorejune25 = pt.sort_index(ascending=False).reset_index()
extprodusersbsscorejune25.columns = extprodusersbsscorejune25.columns.droplevel(1)
extprodusersbsscorejune25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersbsscorejune25


,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,59,0.59322,1455604.103,565284.455,0.38835
1,B,338,0.597633,5956425.544,3586132.601,0.602061
2,C,427,0.765808,7254814.093,4868925.707,0.67113
3,D,389,0.796915,5504587.567,4160363.491,0.755799
4,E,225,0.857778,2518753.282,2134387.548,0.847398
5,F,156,0.858974,1899761.591,1648078.855,0.867519


### Slide 4 lower part

In [109]:
# Slide 4 lower part
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extprodusersbsscorejuly25 = pt.sort_index(ascending=False).reset_index()
extprodusersbsscorejuly25.columns = extprodusersbsscorejuly25.columns.droplevel(1)
extprodusersbsscorejuly25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersbsscorejuly25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,69,0.507246,1.349383e+06,666014.364,0.49357
1,B,539,0.58256,9.937223e+06,5751505.789,0.578784
2,C,679,0.696613,1.323840e+07,9166951.525,0.692452
3,D,596,0.765101,9.056578e+06,6670981.581,0.73659
4,E,324,0.83642,4.389423e+06,3612927.036,0.823098
5,F,269,0.855019,3.230952e+06,2780124.665,0.860466


In [110]:
# Existing prod users, BS score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extprodusersbsscoreaug25 = pt.sort_index(ascending=False).reset_index()
extprodusersbsscoreaug25.columns = extprodusersbsscoreaug25.columns.droplevel(1)
extprodusersbsscoreaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extprodusersbsscoreaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,61,0.409836,9.690402e+05,475605.84,0.490801
1,B,439,0.555809,7.984924e+06,4657716.471,0.583314
2,C,539,0.660482,1.051450e+07,6967329.655,0.66264
3,D,470,0.740426,7.198331e+06,5108778.117,0.709717
4,E,251,0.804781,3.134884e+06,2660624.071,0.848715
5,F,187,0.877005,2.253418e+06,1966831.004,0.872821


### bs existing

In [111]:
dt_bs_oop_ex_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_ex_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_41656\125666790.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dt_bs_oop_ex_calc["oop_risk_segment_bs"] = pd.cut(


In [112]:
# Existing BS users, BS score, June
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extbsusersbsscorejune25 = pt.sort_index(ascending=False).reset_index()
extbsusersbsscorejune25.columns = extbsusersbsscorejune25.columns.droplevel(1)
extbsusersbsscorejune25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extbsusersbsscorejune25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,152,0.526316,2.576747e+06,1061524.688,0.411963
1,B,1074,0.664804,1.693637e+07,10504792.384,0.620251
2,C,1400,0.782857,2.224695e+07,15883131.354,0.713946
3,D,1133,0.80759,1.557555e+07,11352586.871,0.728872
4,E,613,0.871126,7.459715e+06,6221000.157,0.833946
5,F,479,0.885177,5.812191e+06,4873897.283,0.838565


In [113]:
# Existing BS users, BS score, July
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extbsusersbsscorejuly25 = pt.sort_index(ascending=False).reset_index()
extbsusersbsscorejuly25.columns = extbsusersbsscorejuly25.columns.droplevel(1)
extbsusersbsscorejuly25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extbsusersbsscorejuly25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,214,0.481308,3.615562e+06,1369825.34,0.378869
1,B,1597,0.65623,2.496335e+07,14740677.553,0.590493
2,C,1742,0.745695,2.643518e+07,18331914.13,0.693467
3,D,1056,0.801136,1.499830e+07,10925836.096,0.728472
4,E,490,0.855102,6.060241e+06,4995139.09,0.824248
5,F,414,0.874396,4.936128e+06,4161274.687,0.843024


In [114]:
# Existing BS users, BS score, Aug
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)
extbsusersbsscoreaug25 = pt.sort_index(ascending=False).reset_index()
extbsusersbsscoreaug25.columns = extbsusersbsscoreaug25.columns.droplevel(1)
extbsusersbsscoreaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
extbsusersbsscoreaug25

,OOP_Risk_Group,Count,Bad_rate,OS_bal_as_of_resignation_date,OS_bal_flag_bad_as_of_today,Outstanding_balance_bad_rate
0,A,200,0.47,3.119012e+06,1269770.666,0.407107
1,B,1390,0.625899,2.124189e+07,12427792.979,0.585061
2,C,1563,0.730006,2.358223e+07,16355325.838,0.693545
3,D,901,0.782464,1.299697e+07,9408982.602,0.723937
4,E,399,0.834586,5.026586e+06,4290711.041,0.853603
5,F,340,0.9,4.074340e+06,3598772.592,0.883277


# Attrition

In [115]:
# BS Attrition
sql_query_attrition = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_feb26_20260301_attrition`
"""

dt_bs_attr = client.query(sql_query_attrition).to_dataframe()

In [116]:
print(f"The shape of the dataframe backscored attrition data is:\t  {dt_bs_attr.shape}")

The shape of the dataframe backscored attrition data is:	  (1932839, 8)


In [117]:
dd.query("""select ee_customer_id, count(ee_customer_id)cnt from dt_bs_attr group by 1 having cnt > 1;""").to_df()

,ee_customer_id,cnt
0,1244779,19
1,1246067,19
2,1246136,19
3,1251382,19
4,1252279,15
...,...,...
170619,1604963,2
170620,1607768,2
170621,1611055,2
170622,1613319,2


In [118]:
dd.query("""select * from dt_bs_attr where ee_customer_id = '1202344';""").to_df()

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment
0,1202344,2024-08-01,202407,1,0.0,12.0,10.0,Low
1,1202344,2024-09-01,202407,1,0.0,12.0,10.0,Low
2,1202344,2024-10-01,202407,1,0.0,12.0,10.0,Low
3,1202344,2024-11-01,202407,0,0.0,12.0,9.0,Average
4,1202344,2025-01-01,202407,0,0.0,12.0,10.0,Low
5,1202344,2024-12-01,202407,0,0.0,12.0,9.0,Average
6,1202344,2025-02-01,202407,0,0.0,12.0,10.0,Low
7,1202344,2025-03-01,202407,0,0.0,12.0,10.0,Low
8,1202344,2025-05-01,202407,0,0.0,12.0,9.0,Average
9,1202344,2025-04-01,202407,0,0.0,12.0,9.0,Average


In [119]:
dt_bs_attr['ee_customer_id'] = dt_bs_attr['ee_customer_id'].astype('str')

dt_bs_attr["calc_date"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce")
dt_bs_attr["calc_date_correct"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_attr['calc_date_ym'] = dt_bs_attr['calc_date_correct'].dt.year*100 + dt_bs_attr['calc_date_correct'].dt.month

In [120]:
new_1m = dt_bs_attr['ee_onboarding_month'] == dt_bs_attr['calc_date_ym']
dt_bs_attr['is_new_customer_flag_1m'] = new_1m.astype('int')

In [121]:
dt_bs_attr.sample(5)

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m
1816507,1234347,2026-02-01,202407,0,0.0,12.0,8.0,Average,2026-01-01,202601,0
1093249,1157193,2025-07-01,202402,0,0.0,12.0,12.0,Low,2025-06-01,202506,0
190949,1035378,2024-04-01,202304,0,0.0,12.0,10.0,Low,2024-03-01,202403,0
1636823,1231802,2026-01-01,202407,0,0.0,12.0,inf,Very low,2025-12-01,202512,0
1679361,882545,2026-01-01,202209,0,0.0,12.0,6.0,High,2025-12-01,202512,0


In [122]:
# dt_res = pd.read_pickle(
#     generate_bucket_url('DC/Tendo_Model_Monitoring_Data/resignation_data_20260330.pkl', GS_BUCKET)
# )

print(f"The shape of the resignation data downloaded from the cloud is:\t {dt_res.shape}")
dt_res.sample(5)

The shape of the resignation data downloaded from the cloud is:	 (197787, 2)


,ee_customer_id,ee_resignation_date_correct
687563,1369705,2026-01-27
615735,1214145,NaT
590795,1481364,NaT
1751314,1141040,2024-09-12
644431,1041092,2024-06-04


## Distribution metrics

### prod new

In [123]:
dt_prod_api.shape

(165391, 15)

In [124]:
dt_prod_api.sample(5)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,ee_onboarding_date,oop_target,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,onb_rd_diff,onboarding_date_ym,run_date_ym
145975,1360640,2025-08-21,5,5,0.590629,A,2025-11-22,<NA>,0.588173,NaN,NaN,10829.880,93.0,202511.0,202508
8064,1583308,2025-10-19,3,3,0.306979,E,2025-10-19,<NA>,0.284131,11293.787,11293.787,11293.787,0.0,202510.0,202510
32855,1675067,2026-01-08,3,3,0.358539,E,2026-01-08,<NA>,0.358539,NaN,NaN,0.000,0.0,202601.0,202601
82043,1642490,2025-12-06,4,4,0.537492,C,2025-12-20,<NA>,0.537492,NaN,NaN,2923.173,14.0,202512.0,202512
136537,1179872,2025-07-26,6,6,0.585581,A,NaT,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,202507


In [125]:
dt_prod_api_attr = dt_prod_api.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [126]:
print(f"The shape of the dt_prod_api_attr after merging it with dt_res and dt_bs_attr is:\t{dt_prod_api_attr.shape} ")

The shape of the dt_prod_api_attr after merging it with dt_res and dt_bs_attr is:	(165391, 20) 


In [127]:
dt_prod_api_calc = (dt_prod_api_attr
                    .dropna(subset=['ee_onboarding_date'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

In [128]:
print(f"The shape of dt_prod_api_calc after dropping duplicates and null values in ee_onboarding_date is:\t{dt_prod_api_calc.shape}")

The shape of dt_prod_api_calc after dropping duplicates and null values in ee_onboarding_date is:	(11337, 20)


In [129]:
months_diff = (
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.year - dt_prod_api_calc['run_date'].dt.year) * 12 +
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.month - dt_prod_api_calc['run_date'].dt.month)
)

dt_prod_api_calc['time_to_attrition'] = np.where(dt_prod_api_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_api_calc['attrition_event'] = dt_prod_api_calc['ee_resignation_date_correct'].notna().astype('int')

In [130]:
dt_prod_api_calc['score_attr_corrected'] = dt_prod_api_calc['score_attr'].replace(np.inf, 15)
dt_prod_api_calc['attrition_score_prod'] = dt_prod_api_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_api_calc['attrition_risk_segment_prod'] = dt_prod_api_calc['attrition_score_prod'].replace(mapping_dict)

## Slide 5

### Slide 5 upper part

In [131]:
# Slide 5 upper part
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

attrnewprodusersprodscorejuneaug25 = pt.loc[order].reset_index()
# attrnewprodusersprodscorejuneaug25.columns = attrnewprodusersprodscorejuneaug25.columns.droplevel(1)
# extbsusersbsscoreaug25.columns = ['OOP_Risk_Group', 'Count', 'Bad_rate', 'OS_bal_as_of_resignation_date', 'OS_bal_flag_bad_as_of_today', 'Outstanding_balance_bad_rate']
attrnewprodusersprodscorejuneaug25 = attrnewprodusersprodscorejuneaug25[['attrition_risk_segment_prod', 'count', 'attrition_event', 'attrition_score_prod', 'time_to_attrition']]
attrnewprodusersprodscorejuneaug25

,attrition_risk_segment_prod,count,attrition_event,attrition_score_prod,time_to_attrition
0,Very low,44,0.227273,15.000000,0.000000
1,Low,185,0.178378,10.254054,4.333333
2,Average,159,0.188679,8.641509,4.433333
3,High,1080,0.214815,4.450000,3.586207
4,Very high,638,0.252351,2.929467,3.422360


### Slide 5 lower part

In [132]:
# Slide 5 lower part
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrnewprodusersbsscorejuneaug25 = pt.loc[order].reset_index()
attrnewprodusersbsscorejuneaug25 = attrnewprodusersbsscorejuneaug25[['score_attr_segment', 'count', 'attrition_event', 'score_attr_corrected', 'time_to_attrition']]
attrnewprodusersbsscorejuneaug25

,score_attr_segment,count,attrition_event,score_attr_corrected,time_to_attrition
0,Very low,264,0.208333,15.000000,4.690909
1,Low,216,0.152778,10.342593,3.666667
2,Average,615,0.188618,7.912195,4.267241
3,High,890,0.226966,5.174157,4.183168
4,Very high,65,0.246154,2.953846,4.125000


### bs new

In [133]:
dt_bs_attr.sample(5)

,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m
336225,1208429,2024-08-01,202406,1,1.0,11.0,6.0,High,2024-07-01,202407,0
774522,1365381,2025-05-01,202504,1,1.0,3.0,10.0,Low,2025-04-01,202504,1
257140,1209260,2024-09-01,202406,1,0.0,12.0,8.0,Average,2024-08-01,202408,0
1587130,1473234,2025-12-01,202507,0,0.0,12.0,8.0,Average,2025-11-01,202511,0
16304,1067834,2024-01-01,202306,0,1.0,0.0,inf,Very low,2023-12-01,202312,0


In [134]:
for col in dt_bs_attr.select_dtypes(include='datetime').columns:
    dt_bs_attr[col] = dt_bs_attr[col].astype('datetime64[s]')

for col in dt_res.select_dtypes(include='datetime').columns:
    dt_res[col] = dt_res[col].astype('datetime64[s]')

In [135]:
# chunk_size = 500_000
# chunks = []

# for i in range(0, len(dt_bs_attr), chunk_size):
#     chunk = dt_bs_attr.iloc[i:i+chunk_size].merge(
#         dt_res, how='left', on='ee_customer_id'
#     )
#     chunks.append(chunk)

# dt_bs_attr_dev = pd.concat(chunks, ignore_index=True)

In [136]:
dt_bs_attr_dev = dt_bs_attr.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
)

In [137]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_1m == 1').copy()

In [138]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [139]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [140]:
# New BS users, BS score, June-Aug
df = dt_bs_attr_dev_calc.query('ee_onboarding_month >= 202506 & ee_onboarding_month <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

attrnewbsuserbsscorejuneaug25 = pt.loc[order].reset_index()
attrnewbsuserbsscorejuneaug25 = attrnewbsuserbsscorejuneaug25[['score_attr_segment', 'count', 'attrition_event', 'score_attr_corrected', 'time_to_attrition']]
attrnewbsuserbsscorejuneaug25

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_41656\3184220800.py:2: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  df = dt_bs_attr_dev_calc.query('ee_onboarding_month >= 202506 & ee_onboarding_month <= 202508').copy()


,score_attr_segment,count,attrition_event,score_attr_corrected,time_to_attrition
0,Very low,1909,0.137245,15.000000,4.843511
1,Low,2814,0.125800,10.472637,4.437853
2,Average,5862,0.210167,8.138690,4.420455
3,High,2918,0.261480,5.473955,4.159895
4,Very high,438,0.461187,2.582192,4.272277


### prod existing

In [141]:
dt_prod_batch.shape

(406093, 16)

In [142]:
dt_prod_batch.head()

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod,run_date_ym,ee_onboarding_date,oop_target,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date,calc_date_ym,score_oop,onboarding_date_ym,onb_rd_diff
0,927924,2025-10-16,Low,10,0.762006,A,202510,2022-11-16,<NA>,NaN,NaN,NaN,202510.0,0.762006,202211,1065
1,1044039,2025-10-16,Low,10,0.748563,A,202510,2023-05-16,<NA>,NaN,NaN,2585.732,202510.0,0.631576,202305,884
2,910677,2025-10-16,Low,10,0.726238,A,202510,2022-10-16,0,NaN,NaN,NaN,202510.0,0.723239,202210,1096
3,1047238,2025-10-16,Low,10,0.707271,A,202510,2023-05-16,<NA>,NaN,NaN,28541.083,202510.0,0.735159,202305,884
4,1066614,2025-10-16,Low,10,0.739006,A,202510,2023-06-16,<NA>,NaN,NaN,0.000,202510.0,0.739006,202306,853


In [143]:
# import polars as pl

# df_prod = pl.from_pandas(dt_prod_batch)
# df_res = pl.from_pandas(dt_res)
# df_bs = pl.from_pandas(dt_bs_attr[['ee_customer_id', 'calc_date_ym', 
#                                     'score_attr', 'score_attr_segment', 
#                                     'is_new_customer_flag_1m']])

# result = (
#     df_prod
#     .join(df_res, on='ee_customer_id', how='left')
#     .join(df_bs, left_on=['ee_customer_id', 'run_date_ym'], 
#           right_on=['ee_customer_id', 'calc_date_ym'], how='left')
# )

In [144]:
dd.query("""select ee_customer_id, count(ee_customer_id) cnt from dt_res group by 1 having cnt > 1 order by cnt desc;""").to_df()

,ee_customer_id,cnt


In [145]:
dd.query("""select * from dt_res where ee_customer_id = '1065228';""").to_df()

,ee_customer_id,ee_resignation_date_correct
0,1065228,NaT


In [146]:
dd.query("""select ee_customer_id, count(ee_customer_id)cnt from dt_res group by 1 having cnt > 1 order by 2 desc;""").to_df()

,ee_customer_id,cnt


In [147]:
dt_prod_batch_attr = dt_prod_batch.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

print(f"The shape of dt_prod_batch_attr after merging dt_prod_batch with dt_res and dt_bs_attr is:\t {dt_prod_batch_attr.shape}")

The shape of dt_prod_batch_attr after merging dt_prod_batch with dt_res and dt_bs_attr is:	 (406093, 21)


In [148]:
dt_prod_batch_attr.dropna(subset=['ee_onboarding_date']).query('run_date_ym == 202507')['ee_customer_id'].nunique()

35684

In [149]:
dt_prod_batch_calc = dt_prod_batch_attr.dropna(subset=['ee_onboarding_date'])
print(f"The shape of the dt_prod_batch_calc after dropping null values from ee_onboarding_date is:\t {dt_prod_batch_calc.shape}")

The shape of the dt_prod_batch_calc after dropping null values from ee_onboarding_date is:	 (406093, 21)


In [150]:
months_diff = (
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.year - dt_prod_batch_calc['run_date'].dt.year) * 12 +
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.month - dt_prod_batch_calc['run_date'].dt.month)
)

dt_prod_batch_calc['time_to_attrition'] = np.where(dt_prod_batch_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_batch_calc['attrition_event'] = dt_prod_batch_calc['ee_resignation_date_correct'].notna().astype('int')

In [151]:
dt_prod_batch_calc['score_attr_corrected'] = dt_prod_batch_calc['score_attr'].replace(np.inf, 15)
dt_prod_batch_calc['attrition_score_prod'] = dt_prod_batch_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_batch_calc['attrition_risk_segment_prod'] = dt_prod_batch_calc['attrition_score_prod'].replace(mapping_dict)

In [152]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

,attrition_event,attrition_score_prod,time_to_attrition,count
attrition_risk_segment_prod,,,,
Very low,0.136236,15.000000,4.648012,4984
Low,0.128168,10.613582,4.399254,2091
Average,0.178924,8.030628,3.470986,5779
High,0.246425,5.356121,3.697066,6363
Very high,0.430085,2.891949,3.591133,472


## slide 6

### Slide 6 Upper part

In [153]:
# Slide 6 Upper part
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrextprodusersprodscorejuly25 = pt.loc[order].reset_index()
attrextprodusersprodscorejuly25 = attrextprodusersprodscorejuly25[['attrition_risk_segment_prod', 'count', 'attrition_event', 'attrition_score_prod', 'time_to_attrition']]
attrextprodusersprodscorejuly25


,attrition_risk_segment_prod,count,attrition_event,attrition_score_prod,time_to_attrition
0,Very low,8954,0.128769,15.000000,4.124892
1,Low,5054,0.129600,10.443213,3.760305
2,Average,13288,0.164208,8.105433,3.281852
3,High,8261,0.236896,5.500787,3.390393
4,Very high,127,0.385827,3.000000,3.306122


In [154]:
# Existing prod users, prod score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrextprodusersprodscoreaug25 = pt.loc[order].reset_index()
attrextprodusersprodscoreaug25

,attrition_risk_segment_prod,attrition_event,attrition_score_prod,time_to_attrition,count
0,Very low,0.113253,15.000000,3.506424,8247
1,Low,0.109626,10.428627,3.127208,5163
2,Average,0.141134,8.115601,3.052438,13512
3,High,0.203967,5.511141,3.063025,8168
4,Very high,0.274725,3.000000,2.600000,91


In [155]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]
attrextproduserbscorejun25 = pt.loc[order].reset_index()
attrextproduserbscorejun25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.129559,15.000000,4.988796,5511
1,Low,0.146822,10.454143,4.928767,2486
2,Average,0.172463,8.119974,4.298797,6268
3,High,0.247025,5.393927,3.992525,4874
4,Very high,0.421429,2.907143,4.398305,280


### Slide 6 lower part

In [156]:
# Slide 6 lower part
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextproduserbsscorejuly25 = pt.loc[order].reset_index()
attrextproduserbsscorejuly25 = attrextproduserbsscorejuly25[['score_attr_segment', 'count', 'attrition_event', 'score_attr_corrected', 'time_to_attrition']]
attrextproduserbsscorejuly25

,score_attr_segment,count,attrition_event,score_attr_corrected,time_to_attrition
0,Very low,8932,0.123936,15.000000,4.482385
1,Low,5056,0.119066,10.430775,4.455150
2,Average,13023,0.147508,8.126776,4.262363
3,High,7877,0.208201,5.516059,4.025000
4,Very high,103,0.320388,3.000000,3.666667


In [157]:
# Existing prod users, BS score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextprodusersbsscoreaug25 = pt.loc[order].reset_index()
attrextprodusersbsscoreaug25


,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.105707,15.000000,3.897110,8183
1,Low,0.103109,10.427784,4.000000,5082
2,Average,0.130833,8.126182,3.879014,13330
3,High,0.180160,5.525737,3.565881,8004
4,Very high,0.271605,3.000000,2.954545,81


### bs existing

In [158]:
print(f"The shape of dt_bs_attr_dev_calc is:\t{dt_bs_attr_dev_calc.shape}")
dt_bs_attr_dev_calc.head()

The shape of dt_bs_attr_dev_calc is:	(156944, 15)


,ee_customer_id,calc_date,ee_onboarding_month,is_new_customer_flag_3m,target_event,target_duration_dev,score_attr,score_attr_segment,calc_date_correct,calc_date_ym,is_new_customer_flag_1m,ee_resignation_date_correct,time_to_attrition,attrition_event,score_attr_corrected
23,1119478,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,2025-02-13,14.0,1,2.0
25,1121063,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,NaT,NaN,0,2.0
26,1122569,2024-01-01,202312,1,1.0,6.0,2.0,Very high,2023-12-01,202312,1,2024-07-05,7.0,1,2.0
30,1126761,2024-01-01,202312,1,0.0,12.0,2.0,Very high,2023-12-01,202312,1,NaT,NaN,0,2.0
33,1127534,2024-01-01,202312,1,1.0,0.0,2.0,Very high,2023-12-01,202312,1,2024-01-06,1.0,1,2.0


In [159]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_3m == 0').copy()

C:\Users\Dwaipayan\AppData\Local\Temp\ipykernel_41656\3206105625.py:1: RuntimeWarning: Engine has switched to 'python' because numexpr does not support extension array dtypes. Please set your engine to python manually.
  dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_3m == 0').copy()


In [160]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [161]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [162]:
# Existing BS users, BS score, June
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextbsusersbsscorejun25 = pt.loc[order].reset_index()
attrextbsusersbsscorejun25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.132850,15.000000,4.690421,19330
1,Low,0.151732,10.524227,4.661417,7533
2,Average,0.178092,8.126963,4.276794,17446
3,High,0.252867,5.431802,4.296237,12295
4,Very high,0.400763,2.925573,4.119048,524


In [163]:
# Existing BS users, BS score, July
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextbsusersbsscorejul25 = pt.loc[order].reset_index()
attrextbsusersbsscorejul25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.116371,15.000000,4.381773,20933
1,Low,0.121278,10.486485,4.535757,13836
2,Average,0.168933,8.165043,4.225737,27508
3,High,0.229262,5.511287,4.086681,14442
4,Very high,0.369565,2.932609,3.652941,460


In [164]:
# Existing BS users, BS score, August
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

attrextbsusersbsscoreaug25 = pt.loc[order].reset_index()
attrextbsusersbsscoreaug25

,score_attr_segment,attrition_event,score_attr_corrected,time_to_attrition,count
0,Very low,0.101009,15.000000,3.984665,21305
1,Low,0.107008,10.490364,4.010076,14840
2,Average,0.154484,8.163943,3.793163,29919
3,High,0.203840,5.520497,3.613103,15051
4,Very high,0.366133,2.876430,3.043750,437


In [165]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

# Count of Onboarded customer as per scorecard 2.0 - New logic Three months from Onboarding

In [166]:
sq = """ 
select 
date(user_timelines.first_account_activated_at) as ee_onboarding_date,
format_date('%Y-%m', date(user_timelines.first_account_activated_at)) Onboarding_month,
t1.employee_id,
t1.oop_risk_segment,
  from prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api t1
  inner join tendopay_raw.user_timelines on cast(user_timelines.user_id as string) = cast(t1.employee_id as string)
   where user_timelines.first_account_activated_at >= '2025-06-01'
        and user_timelines.first_account_activated_at < '2026-01-01'
     qualify row_number() over(partition by t1.employee_id order by t1.run_date) = 1;
"""
df1 = client.query(sq).to_dataframe()

In [167]:
df1.groupby(['Onboarding_month', 'oop_risk_segment']).agg({'employee_id':'nunique'}).reset_index()

,Onboarding_month,oop_risk_segment,employee_id
0,2025-06,A,28
1,2025-06,B,28
2,2025-06,C,34
3,2025-06,D,6
4,2025-06,E,5
5,2025-06,F,7
6,2025-07,A,489
7,2025-07,B,289
8,2025-07,C,141
9,2025-07,D,39


In [168]:

sq = """ 
select user_id, tag_id, tag_name, created_at resignation_date, deleted_at 
from prj-prod-dataplatform.tendopay_raw.user_tag
where tag_id = 100
and deleted_at is null;
"""
userdf = client.query(sq).to_dataframe()


In [169]:
sq = """select user_id, target_maturity_flag, target , ee_resignation_date_correct 
from prj-prod-dataplatform.tendo_mart.tendo_collection_target_master"""
df_target = client.query(sq).to_dataframe()
df_target.shape

(46346, 4)

In [170]:
for col in df1.columns:
    col_type = str(df1[col].dtype).lower()
    
    # Handle date/time types
    if 'dbdate' in col_type or 'date' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col])
        except:
            pass
    elif 'dbtime' in col_type or 'time' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col], format='%H:%M:%S').dt.time
        except:
            pass
    elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
        try:
            df1[col] = pd.to_datetime(df1[col])
        except:
            pass

df2 = dd.query("""select * from df1 
         left join userdf on userdf.user_id = df1.employee_id
            left join df_target on cast(df_target.user_id as string) = cast (df1.employee_id as string)
                   """).to_df()       

In [171]:
df2.head()

,ee_onboarding_date,Onboarding_month,employee_id,oop_risk_segment,user_id,tag_id,tag_name,resignation_date,deleted_at,user_id_1,target_maturity_flag,target,ee_resignation_date_correct
0,2025-07-05,2025-07,1409198,A,1409198,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1409198,1,0,2025-12-29 05:30:00+05:30
1,2025-09-20,2025-09,1557099,C,1557099,100,FREEZE_PERMANENT,2026-01-05 20:22:23+05:30,NaT,1557099,1,1,2025-12-17 05:30:00+05:30
2,2025-11-22,2025-11,1588636,C,1588636,100,FREEZE_PERMANENT,2026-01-07 16:28:11+05:30,NaT,1588636,1,0,2025-12-30 05:30:00+05:30
3,2025-06-12,2025-06,1454348,B,1454348,100,FREEZE_PERMANENT,2026-01-07 16:28:11+05:30,NaT,1454348,1,0,2025-12-30 05:30:00+05:30
4,2025-09-19,2025-09,1553870,C,1553870,100,FREEZE_PERMANENT,2026-01-07 16:28:11+05:30,NaT,1553870,1,0,2025-12-15 05:30:00+05:30


In [172]:
dd.query("""select * from df2 where date(resignation_date) != date(ee_resignation_date_correct);""").to_df()

,ee_onboarding_date,Onboarding_month,employee_id,oop_risk_segment,user_id,tag_id,tag_name,resignation_date,deleted_at,user_id_1,target_maturity_flag,target,ee_resignation_date_correct
0,2025-07-05,2025-07,1409198,A,1409198,100,FREEZE_PERMANENT,2025-12-30 03:38:09+05:30,NaT,1409198,1,0,2025-12-29 05:30:00+05:30
1,2025-09-20,2025-09,1557099,C,1557099,100,FREEZE_PERMANENT,2026-01-05 20:22:23+05:30,NaT,1557099,1,1,2025-12-17 05:30:00+05:30
2,2025-11-22,2025-11,1588636,C,1588636,100,FREEZE_PERMANENT,2026-01-07 16:28:11+05:30,NaT,1588636,1,0,2025-12-30 05:30:00+05:30
3,2025-06-12,2025-06,1454348,B,1454348,100,FREEZE_PERMANENT,2026-01-07 16:28:11+05:30,NaT,1454348,1,0,2025-12-30 05:30:00+05:30
4,2025-09-19,2025-09,1553870,C,1553870,100,FREEZE_PERMANENT,2026-01-07 16:28:11+05:30,NaT,1553870,1,0,2025-12-15 05:30:00+05:30
...,...,...,...,...,...,...,...,...,...,...,...,...,...
535,2025-06-03,2025-06,1447152,A,1447152,100,FREEZE_PERMANENT,2025-07-19 00:10:25+05:30,NaT,1447152,1,1,2025-07-18 05:30:00+05:30
536,2025-06-05,2025-06,1260893,B,1260893,100,FREEZE_PERMANENT,2025-07-29 21:52:44+05:30,NaT,1260893,1,1,2025-07-20 05:30:00+05:30
537,2025-07-06,2025-07,1474493,A,1474493,100,FREEZE_PERMANENT,2025-08-20 21:01:53+05:30,NaT,1474493,1,1,2025-08-04 05:30:00+05:30
538,2025-07-11,2025-07,1395542,A,1395542,100,FREEZE_PERMANENT,2025-09-02 08:30:33+05:30,NaT,1395542,1,0,2025-08-12 05:30:00+05:30


In [173]:
data = dd.query("""select Onboarding_month, oop_risk_segment,
         count(distinct employee_id) as cnt_onboarded_with_sc2,
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) as cnt_left_job_within_Jan_2026,
         count(distinct case when target_maturity_flag = 1 then df2.employee_id end) as cnt_had_oop_outstanding,
         count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end) as cnt_oop_flag_bad_fpd30,
         round(count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end)/
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) *100, 2) as unit_rate_of_all_who_left_job
         from df2
         group by Onboarding_month, oop_risk_segment
         order by Onboarding_month
         """).to_df()

In [174]:
data

,Onboarding_month,oop_risk_segment,cnt_onboarded_with_sc2,cnt_left_job_within_Jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad_fpd30,unit_rate_of_all_who_left_job
0,2025-06,D,6,1,1,1,100.00
1,2025-06,C,34,3,3,1,33.33
2,2025-06,E,5,1,1,1,100.00
3,2025-06,F,7,0,0,0,NaN
4,2025-06,B,28,5,5,3,60.00
5,2025-06,A,28,4,5,2,50.00
6,2025-07,B,289,26,30,19,73.08
7,2025-07,C,141,24,23,16,66.67
8,2025-07,A,489,41,52,35,85.37
9,2025-07,D,39,5,4,4,80.00


In [175]:
data.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_sc2_0_monthlytilldec25.xlsx", sheet_name='New_Tendo_funnel_scorecard2_0')

In [230]:
data2 = dd.query("""select Onboarding_month,
         count(distinct employee_id) as cnt_onboarded_with_sc2,
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-02-01') then ee_resignation_date_correct end) as cnt_left_job_within_Jan_2026,
         count(distinct case when target_maturity_flag = 1 then df2.employee_id end) as cnt_had_oop_outstanding,
         count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end) as cnt_oop_flag_bad_fpd30,
         round(count(distinct case when target_maturity_flag = 1 and target = 0 then df2.employee_id end)/
         count(distinct case when ee_resignation_date_correct is not null
         and date(ee_resignation_date_correct) < date('2026-03-01') then df2.employee_id end) , 2) as unit_rate_of_all_who_left_job
         from df2
         group by Onboarding_month
         order by 1
         """).to_df()

In [231]:
data2

,Onboarding_month,cnt_onboarded_with_sc2,cnt_left_job_within_Jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad_fpd30,unit_rate_of_all_who_left_job
0,2025-06,108,14,15,8,0.40
1,2025-07,1135,71,122,82,0.44
2,2025-08,863,50,86,60,0.51
3,2025-09,921,59,108,79,0.50
4,2025-10,575,34,45,29,0.39
5,2025-11,951,32,63,43,0.42
6,2025-12,1079,13,27,13,0.29


In [178]:
data2.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_scorecard2_0_monthlydec25.xlsx", sheet_name='New_Tendo_funnel_sc_2_0_rs')

# Peso values of customers Onboarded

## Select the employees who onboarded from June to August 2025

In [179]:
sq = """ 
select distinct
date(user_timelines.first_account_activated_at) as ee_onboarding_date,
format_date('%Y-%m', date(user_timelines.first_account_activated_at)) Onboarding_month,
t1.employee_id,
t1.oop_risk_segment,
  from prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api t1
  inner join tendopay_raw.user_timelines on cast(user_timelines.user_id as string) = cast(t1.employee_id as string)
   where user_timelines.first_account_activated_at >= '2025-06-01'
     and user_timelines.first_account_activated_at < '2026-01-01'
  qualify row_number() over(partition by t1.employee_id order by t1.run_date) = 1;
"""
df1 = client.query(sq).to_dataframe()
print(f"The data for the customers who onboarded between June to August 2025 is: {df1.shape}")

The data for the customers who onboarded between June to August 2025 is: (5632, 4)


In [180]:
df1.groupby('Onboarding_month').agg({'employee_id':'nunique'}).reset_index()

,Onboarding_month,employee_id
0,2025-06,108
1,2025-07,1135
2,2025-08,863
3,2025-09,921
4,2025-10,575
5,2025-11,951
6,2025-12,1079


In [181]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

In [182]:
convert_datetime_columns(df1)

In [183]:
dd.query("""select employee_id, count(employee_id) as emp_count from df1 group by employee_id having emp_count > 1 order by 2 desc;""").to_df()

,employee_id,emp_count


In [184]:
sq = """ 
with frozen_tags AS (
  SELECT
    user_id,
    max(CASE WHEN tag_id IN (39, 100, 101, 102, 103) THEN 1 ELSE 0 END)
      AS frozen_tag
  FROM `tendopay_raw.user_tag`
  GROUP BY user_id
),
employee_info as
(SELECT
    u.id user_id,
    u.created_at sign_up_date,
    ut.first_account_activated_at AS ee_onboarding_date,
    ut.first_account_activated_at approval_date,
    u.employer_id employer_id,
    e.name employer_name,
    ii.employment_date employment_date,
    LENGTH(employment_date) LENGTH_employment_date,
    datetime_diff(
      CAST(ii.created_at AS date),
      CASE  ------original: datetime_diff(cast(ut.first_account_activated_at as date),
        WHEN LENGTH(employment_date) = 6
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 1) AS int64),
              1)
        WHEN LENGTH(employment_date) = 7
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 2) AS int64),
              1)
        WHEN LENGTH(employment_date) = 10
          THEN
            date(
              CAST(substr(employment_date, 1, 4) AS int64),
              CAST(substr(employment_date, 6, 2) AS int64),
              CAST(substr(employment_date, 9, 2) AS int64))
        END,
      day)
      / 365
      AS employer_time,  ----sometimes employment_date is missing, sometimes approval_date (first_account_activated_at) is missing
    u.gender gender,
    u.civil_status civil_status,
    ii.employee_status employment_status,
    CASE
      WHEN ii.income IN ('0-10000') THEN 5000
      WHEN ii.income IN ('10000-20000') THEN 15000
      WHEN ii.income IN ('20000-30000') THEN 25000
      WHEN ii.income IN ('30000-40000') THEN 35000
      WHEN ii.income IN ('40000-50000') THEN 45000
      WHEN ii.income IN ('50000+') THEN 50000
      ELSE CAST(ii.income AS numeric)
      END AS declared_income_num,
    ii.verified_net_income verified_net_income,
    CASE
      WHEN tag.frozen_tag = 1 THEN 'Frozen'
      ELSE 'Not Frozen'
      END Frozen_Status,

      cci.value                                                                               Credit_limit,
       
  FROM `tendopay_raw.users` u
  LEFT JOIN `tendopay_raw.income_info` ii
    ON u.id = ii.user_id
  LEFT JOIN `tendopay_raw.user_timelines` ut
    ON u.id = ut.user_id
  LEFT JOIN `tendopay_raw.employers` e
    ON u.employer_id = CAST(e.id AS string)
  LEFT JOIN frozen_tags tag
    ON u.id = tag.user_id

  LEFT JOIN `tendopay_raw.customer_credit_information` cci on u.id = cci.user_id
  WHERE u.product_type IN ('employer', 'payroll')

  and u.account_activated = 2
  and cci.`key` = 'credit-limit'
)
select user_id, 
sign_up_date,
ee_onboarding_date,
employer_id,
employment_date,
LENGTH_employment_date,
gender,
civil_status,
employment_status,
declared_income_num,
verified_net_income,
Frozen_Status,
Credit_limit,
 from employee_info
;
"""

df_employee = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The data for the employee info is: {df_employee.shape}")


Job ID 3c0b5414-982f-4ef4-92cc-5e77b628e55d successfully executed: 100%|██████████|
Downloading: 100%|██████████|
The data for the employee info is: (196843, 13)


In [185]:
df_employee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196843 entries, 0 to 196842
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   user_id                 196843 non-null  Int64              
 1   sign_up_date            196843 non-null  datetime64[us, UTC]
 2   ee_onboarding_date      196841 non-null  datetime64[us, UTC]
 3   employer_id             196772 non-null  object             
 4   employment_date         196821 non-null  object             
 5   LENGTH_employment_date  196821 non-null  Int64              
 6   gender                  196808 non-null  object             
 7   civil_status            195591 non-null  object             
 8   employment_status       185757 non-null  object             
 9   declared_income_num     196825 non-null  object             
 10  verified_net_income     196687 non-null  object             
 11  Frozen_Status           19

In [186]:
df1.columns

Index(['ee_onboarding_date', 'Onboarding_month', 'employee_id',
       'oop_risk_segment'],
      dtype='object')

In [187]:
df1['user_id'] = df1['employee_id'].astype('int64')

## Merging Onboarded employee from June 2025 to August 2025 with employee info data

In [188]:
data = df1.merge(df_employee, left_on='user_id', right_on='user_id', how='left')
print(f"The shape of the data after merging employee info is: {data.shape}")

The shape of the data after merging employee info is: (5632, 17)


In [189]:
data.head()

,ee_onboarding_date_x,Onboarding_month,employee_id,oop_risk_segment,user_id,sign_up_date,ee_onboarding_date_y,employer_id,employment_date,LENGTH_employment_date,gender,civil_status,employment_status,declared_income_num,verified_net_income,Frozen_Status,Credit_limit
0,2025-07-06,2025-07,1350642,A,1350642,2025-02-26 18:48:39+00:00,2025-07-06 16:45:16+00:00,10159,2025-02,7,Male,Married,probationary,85000.000000000,63000,Not Frozen,75600
1,2025-12-16,2025-12,1369700,B,1369700,2025-03-28 17:46:43+00:00,2025-12-16 14:39:43+00:00,10048,2024-09,7,Female,Single,regular,35000.000000000,18000,Frozen,48762.00
2,2025-09-02,2025-09,1433309,A,1433309,2025-05-16 11:06:54+00:00,2025-09-02 16:25:52+00:00,10107,2025-03,7,Female,Married,regular,70000.000000000,61500,Not Frozen,123000.00
3,2025-07-07,2025-07,1466744,A,1466744,2025-06-26 09:19:44+00:00,2025-07-07 10:48:52+00:00,10161,2024-08,7,Male,Single,regular,20000.000000000,9500,Frozen,16510.00
4,2025-07-02,2025-07,1470831,C,1470831,2025-07-01 09:07:48+00:00,2025-07-02 15:16:17+00:00,10296,2022-09,7,Female,Married,regular,132000.000000000,104000,Frozen,291200.00


In [190]:
def convert_datetime_columns(df):
    for col in df.columns:
        col_type = str(df[col].dtype).lower()
        
        # Handle date/time types
        if 'dbdate' in col_type or 'date' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass
        elif 'dbtime' in col_type or 'time' in col_type:
            try:
                df[col] = pd.to_datetime(df[col], format='%H:%M:%S').dt.time
            except:
                pass
        elif 'dbtimestamp' in col_type or 'timestamp' in col_type:
            try:
                df[col] = pd.to_datetime(df[col])
            except:
                pass

Check duplicates

In [191]:
convert_datetime_columns(data)
dd.query("""select employee_id, count(employee_id) as emp_count from data group by employee_id having emp_count > 1 order by 2 desc;""").to_df()

,employee_id,emp_count


## OOP maturity target merged with Oleh's backscore table

In [234]:
sq = """ 
with b1 as 
(select user_id, target_maturity_flag, target , ee_resignation_date_correct 
from prj-prod-dataplatform.tendo_mart.tendo_collection_target_master
where target_maturity_flag = 1
)
select * from b1 
left join `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal` b2 on cast(b1.user_id as numeric) = cast(b2.ee_customer_id as numeric);
"""
df_target = client.query(sq).to_dataframe()
print(f"The data for the target maturity info is: {df_target.shape}")

The data for the target maturity info is: (36336, 11)


In [235]:
df_target['user_id'] = df_target['user_id'].astype('int64')

Check for duplicates

In [236]:
dd.query("""select user_id, count(user_id) cnt from df_target group by 1 having count(user_id) > 1 order by 2 desc;""").to_df()

,user_id,cnt


In [237]:
data1 = data.merge(df_target, left_on='user_id', right_on='user_id', how='inner')

In [238]:
data1.head()

,ee_onboarding_date_x,Onboarding_month,employee_id,oop_risk_segment,user_id,sign_up_date,ee_onboarding_date_y,employer_id,employment_date,LENGTH_employment_date,gender,civil_status,employment_status,declared_income_num,verified_net_income,Frozen_Status,Credit_limit,target_maturity_flag,target,ee_resignation_date_correct,ee_customer_id,calc_date,target_dev,score_oop,osbal_as_of_resignation_date,osbal_as_of_oop_eligible_date,osbal_as_of_current_date
0,2025-10-09,2025-10,1527949,B,1527949,2025-08-26 11:30:31+00:00,2025-10-09 14:50:08+00:00,10224,2025-06,7,Female,Single,regular,13000.000000000,17000,Frozen,14623.00,1,0,2025-12-15 00:00:00+00:00,1527949,2025-11-01,NaN,0.636563,5481.715,3696.092,3696.092
1,2025-08-30,2025-08,1467200,B,1467200,2025-06-26 19:50:21+00:00,2025-08-30 15:02:35+00:00,10239,2025-05,7,Female,Single,regular,17000.000000000,16000,Frozen,11428.00,1,0,2026-01-25 00:00:00+00:00,1467200,2025-09-01,NaN,0.273579,7118.990,7118.990,7118.990
2,2025-07-03,2025-07,1472948,B,1472948,2025-07-03 11:07:59+00:00,2025-07-03 16:57:10+00:00,10296,2024-02,7,Female,Married,regular,30000.000000000,18000,Frozen,22500,1,0,2025-12-29 00:00:00+00:00,1472948,2025-08-01,0.0,0.401896,18494.608,18494.608,18494.608
3,2025-10-21,2025-10,1576654,B,1576654,2025-10-09 02:14:29+00:00,2025-10-21 11:57:06+00:00,10224,2024-07,7,Female,Single,regular,27000.000000000,22000,Frozen,13718,1,0,2025-11-18 00:00:00+00:00,1576654,2025-11-01,NaN,0.627091,11545.182,11545.182,11545.182
4,2025-09-01,2025-09,1526927,A,1526927,2025-08-25 10:49:31+00:00,2025-09-01 15:35:59+00:00,10107,2025-03,7,Male,Single,regular,60000.000000000,50500,Frozen,20200,1,1,2025-10-14 00:00:00+00:00,1526927,2025-10-01,1.0,0.532888,16989.641,696.461,0.001


In [239]:
dd.query("""select employee_id, count(employee_id) cnt from data1 group by 1 having count(employee_id) > 1 order by 2 desc;""").to_df()

,employee_id,cnt


In [240]:
data3 = dd.query(f""" select Onboarding_month, oop_risk_segment,
         -- count(distinct employee_id) as cnt_onboarded_with_sc2,
         sum(cast(Credit_limit as numeric)) Credit_limit,
         -- count(distinct case when ee_resignation_date_correct is not null then employee_id end) as cnt_resigned_employees,
         -- count(distinct case when target_maturity_flag = 1 then employee_id end) as cnt_had_oop_eligible_cnt,
         -- count(distinct case when target = 0 then employee_id end) as cnt_oop_flag_bad_fpd30,
         -- count(distinct case when target = 1 then employee_id end) as cnt_oop_flag_good_fpd30,
         sum(cast(osbal_as_of_resignation_date as numeric)) as sum_oop_outstanding_resignation_date,
         sum(case when target_maturity_flag = 1 then cast(osbal_as_of_oop_eligible_date as numeric) else 0 end) as osbal_as_of_oop_eligible_date,
         sum(case when target = 0 then cast(coalesce(osbal_as_of_current_date, 0) as numeric) end) as tot_os_principal_amt_from_bad_customers,
         from data1
            group by Onboarding_month, oop_risk_segment
                 order by Onboarding_month, oop_risk_segment
         """
         ).to_df()

In [242]:
data3.head()

,Onboarding_month,oop_risk_segment,Credit_limit,sum_oop_outstanding_resignation_date,osbal_as_of_oop_eligible_date,tot_os_principal_amt_from_bad_customers
0,2025-06,A,118522.0,49193.030,21415.847,6299.818
1,2025-06,B,148429.0,114035.052,109709.934,50587.124
2,2025-06,C,237490.0,78286.440,78286.440,26157.168
3,2025-06,D,12716.0,10659.616,10659.616,10659.616
4,2025-06,E,31282.0,18166.701,18166.701,18166.701


In [243]:
data3.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_scorecard2_0_peso_monthly_dec25.xlsx", sheet_name='New_Tendo_funnel_sc2_0_peso')

In [244]:
data4 = dd.query(f"""select Onboarding_month,
         sum(cast(Credit_limit as numeric)) tot_cl_amt_onboarded_with_2_0,
         --- count(distinct case when ee_resignation_date_correct is not null then employee_id end) as cnt_resigned_employees,
         --- count(distinct case when target_maturity_flag = 1 then employee_id end) as cnt_had_oop_eligible_cnt,
         --- count(distinct case when target = 0 then employee_id end) as cnt_oop_flag_bad_fpd30,
         --- count(distinct case when target = 1 then employee_id end) as cnt_oop_flag_good_fpd30,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-03-01' then cast(osbal_as_of_resignation_date as numeric) else 0 end) as tot_os_principal_amt_as_of_resignation_date,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-03-01' and target_maturity_flag = 1 then cast(osbal_as_of_oop_eligible_date as numeric) else 0 end) as tot_os_principal_amt_as_of_oop_eligible_dt,
         sum(case when ee_resignation_date_correct is not null and date(ee_resignation_date_correct) < '2026-03-01' and target_maturity_flag = 1 and target = 0 then cast(coalesce(osbal_as_of_current_date, 0) as numeric) end) as tot_os_principal_amt_from_bad_customers,
         from data1
                 group by 1
                 order by 1
                   """
         ).to_df()


In [246]:
data4.sort_values(by='Onboarding_month')

,Onboarding_month,tot_cl_amt_onboarded_with_2_0,tot_os_principal_amt_as_of_resignation_date,tot_os_principal_amt_as_of_oop_eligible_dt,tot_os_principal_amt_from_bad_customers
0,2025-06,548439.0,233810.413,201708.112,111870.427
1,2025-07,3383583.5,1299780.094,1189220.744,885792.936
2,2025-08,2216971.0,825895.718,893610.671,743934.593
3,2025-09,2500398.0,1034579.729,1039656.446,759095.109
4,2025-10,1652221.0,643626.572,593696.002,348976.984
5,2025-11,1658608.0,431380.161,463407.448,406528.202
6,2025-12,725192.0,163254.912,208188.235,123785.554


In [203]:
data4.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_New_Tendo_Funnel_scorecard2_0_peso_dec25.xlsx", sheet_name='New_Tendo_funnel_sc2_0_peso')

# For v1

In [225]:
sq = """
WITH first_success AS (
  SELECT
    tendopay_user_id AS user_id,
    MIN(DATE(created_at)) AS onboarding_date
  FROM tendopay_raw.payment_responses
  WHERE status IN ('PTOK','CTOK')
  GROUP BY tendopay_user_id
),

-- ScoreCard 1.0 user level (risk segment)
scorecard_1_0 AS (
  WITH data_preparation AS (
    SELECT
      u.id AS user_id,
      ii.employment_date,
      DATETIME_DIFF(
        CAST(ii.created_at AS DATE),
        CASE
          WHEN LENGTH(ii.employment_date)=6  THEN SAFE.PARSE_DATE('%Y-%m',    ii.employment_date)
          WHEN LENGTH(ii.employment_date)=7  THEN SAFE.PARSE_DATE('%Y-%m',    ii.employment_date)
          WHEN LENGTH(ii.employment_date)=10 THEN SAFE.PARSE_DATE('%Y-%m-%d', ii.employment_date)
        END,
        DAY
      ) / 365 AS employer_time,
      u.gender,
      u.civil_status,
      CASE
        WHEN ii.income IN ('0-10000')     THEN 5000
        WHEN ii.income IN ('10000-20000') THEN 15000
        WHEN ii.income IN ('20000-30000') THEN 25000
        WHEN ii.income IN ('30000-40000') THEN 35000
        WHEN ii.income IN ('40000-50000') THEN 45000
        WHEN ii.income IN ('50000+')      THEN 50000
        ELSE CAST(ii.income AS NUMERIC)
      END AS declared_income_num
    FROM prj-prod-dataplatform.tendopay_raw.users u
    LEFT JOIN prj-prod-dataplatform.tendopay_raw.income_info ii
      ON u.id = ii.user_id
      left join `tendopay_raw.user_timelines` ut
      on u.id = ut.user_id
    WHERE u.product_type IN ('employer') 
    and ut.first_account_activated_at is not null

  ),

  scoring AS (
    SELECT
      *,
      (
        CASE
          WHEN employer_time < 0.55 THEN 98
          WHEN employer_time < 2.35 THEN 123
          WHEN employer_time < 3.85 THEN 139
          WHEN employer_time >= 3.85 THEN 183
          ELSE 119
        END
        + CASE
            WHEN gender = 'Female' THEN 120
            WHEN gender IN ('Male','not specified') THEN 118
            ELSE 119
          END
        + CASE
            WHEN declared_income_num < 21200 THEN 119
            WHEN declared_income_num < 26525 THEN 110
            WHEN declared_income_num < 33050 THEN 118
            WHEN declared_income_num >= 33050 THEN 146
            ELSE 119
          END
        + CASE
            WHEN civil_status IN ('Divorced','Married','Widowed') THEN 127
            WHEN civil_status = 'Single' THEN 117
            ELSE 119
          END
      ) AS score
    FROM data_preparation
  )

  SELECT DISTINCT
    user_id,
    CASE
      WHEN score > 532 THEN 'A'
      WHEN score > 492 THEN 'B'
      WHEN score > 478 THEN 'C'
      WHEN score > 468 THEN 'D'
      WHEN score > 452 THEN 'E'
      ELSE 'F'
    END AS risk_segment
  FROM scoring
),

-- onboarding users with SC 1.0
master_cohort AS (
  SELECT
    f.user_id,
    f.onboarding_date,
    sc.risk_segment
  FROM first_success f
  INNER JOIN scorecard_1_0 sc
    ON f.user_id = sc.user_id
  WHERE f.onboarding_date BETWEEN DATE '2025-06-01' AND DATE '2025-12-31'
),

-- resignation logic
ordered_repayments AS (
  SELECT
    user_id,
    vendor_id,
    repaid_at,
    LEAD(vendor_id) OVER (PARTITION BY user_id ORDER BY repaid_at) AS next_vendor
  FROM tendopay_raw.customer_repayment_responses
  WHERE vendor_id IN ('TPSD','TPFP')
    AND repaid_at IS NOT NULL
),

resignation AS (
  SELECT
    user_id,
    MAX(DATE_ADD(DATE(repaid_at), INTERVAL 1 MONTH)) AS resignation_date
  FROM ordered_repayments
  WHERE vendor_id='TPSD'
    AND next_vendor='TPFP'
  GROUP BY user_id
),


oop_data AS (
  SELECT
    ee_customer_id AS user_id,
    osbal_as_of_resignation_date
  FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
),

oop_target AS (
  SELECT
    user_id,
    `target`, target_maturity_flag,ee_resignation_date_correct
  FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
  WHERE target_maturity_flag = 1

)

-- final
SELECT
  FORMAT_DATE('%Y-%m', mc.onboarding_date) AS onboarding_month,
  --mc.risk_segment,

  COUNT(DISTINCT mc.user_id) AS cnt_onboarded_with_1_0,

  COUNT(DISTINCT CASE WHEN date(tgt.ee_resignation_date_correct) <= DATE '2026-03-01' THEN mc.user_id END) AS cnt_left_job_within_jan_2026,

  COUNT(DISTINCT CASE WHEN   tgt.target_maturity_flag = 1 THEN mc.user_id END) AS cnt_had_oop_outstanding,


  COUNT(DISTINCT CASE WHEN  tgt.target = 0 THEN mc.user_id END) AS cnt_oop_flag_bad,

  round(SAFE_DIVIDE(
    COUNT(DISTINCT CASE
    WHEN  tgt.target = 0
    THEN mc.user_id
  END),
    COUNT(DISTINCT CASE
    WHEN
   target_maturity_flag = 1
    THEN mc.user_id 
  END)
  ) ,2) AS unit_rate_of_all_who_left_job

FROM master_cohort mc
LEFT JOIN resignation r
  ON mc.user_id = r.user_id
LEFT JOIN oop_data oop
  ON mc.user_id = oop.user_id
LEFT JOIN oop_target tgt
  ON cast(mc.user_id as string) = tgt.user_id

GROUP BY  onboarding_month--,  mc.risk_segment
ORDER BY --mc.risk_segment,
onboarding_month;
"""

df2v1 = client.query(sq).to_dataframe()
df2v1
 

,onboarding_month,cnt_onboarded_with_1_0,cnt_left_job_within_jan_2026,cnt_had_oop_outstanding,cnt_oop_flag_bad,unit_rate_of_all_who_left_job
0,2025-06,5302,542,545,361,0.66
1,2025-07,4567,458,468,311,0.66
2,2025-08,4039,330,337,216,0.64
3,2025-09,8610,628,642,449,0.70
4,2025-10,12346,964,984,736,0.75
5,2025-11,7892,488,508,370,0.73
6,2025-12,8592,158,185,96,0.52


In [205]:
df2v1.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_v1pesorateunit.xlsx", sheet_name='v1unitpesorate')

In [224]:
sq = """
   WITH first_success AS (
    SELECT
        tendopay_user_id AS user_id,
        MIN(DATE(created_at)) AS onboarding_date
    FROM tendopay_raw.payment_responses
    WHERE status IN ('PTOK','CTOK')
    GROUP BY tendopay_user_id
),

onboarded_cohort AS (
    SELECT *
    FROM first_success
    WHERE onboarding_date BETWEEN DATE '2025-06-01'
                              AND DATE '2025-12-31'
),

-- Build Score + Risk Segment
scorecard_1_0 AS (

    WITH data_preparation AS (
        SELECT
            u.id AS user_id,
            DATETIME_DIFF(
                CAST(ii.created_at AS DATE),
                SAFE.PARSE_DATE('%Y-%m-%d', ii.employment_date),
                DAY
            ) / 365 AS employer_time,
            u.gender,
            u.civil_status,
            CASE
                WHEN ii.income IN ('0-10000') THEN 5000
                WHEN ii.income IN ('10000-20000') THEN 15000
                WHEN ii.income IN ('20000-30000') THEN 25000
                WHEN ii.income IN ('30000-40000') THEN 35000
                WHEN ii.income IN ('40000-50000') THEN 45000
                WHEN ii.income IN ('50000+') THEN 50000
                ELSE CAST(ii.income AS NUMERIC)
            END AS declared_income_num
        FROM prj-prod-dataplatform.tendopay_raw.users u
        LEFT JOIN prj-prod-dataplatform.tendopay_raw.income_info ii
            ON u.id = ii.user_id
            left join `tendopay_raw.user_timelines` ut
      on u.id = ut.user_id
        WHERE u.product_type IN ('employer')
        and ut.first_account_activated_at is not null
    ),

    scoring AS (
        SELECT
            user_id,
            (
                CASE
                    WHEN employer_time < 0.55 THEN 98
                    WHEN employer_time < 2.35 THEN 123
                    WHEN employer_time < 3.85 THEN 139
                    WHEN employer_time >= 3.85 THEN 183
                    ELSE 119
                END
              + CASE
                    WHEN gender='Female' THEN 120
                    ELSE 118
                END
              + CASE
                    WHEN declared_income_num < 21200 THEN 119
                    WHEN declared_income_num < 26525 THEN 110
                    WHEN declared_income_num < 33050 THEN 118
                    WHEN declared_income_num >= 33050 THEN 146
                    ELSE 119
                END
              + CASE
                    WHEN civil_status IN ('Divorced','Married','Widowed') THEN 127
                    WHEN civil_status='Single' THEN 117
                    ELSE 119
                END
            ) AS score
        FROM data_preparation
    )

    SELECT
        user_id,
        CASE
            WHEN score > 532 THEN 'A'
            WHEN score > 492 THEN 'B'
            WHEN score > 478 THEN 'C'
            WHEN score > 468 THEN 'D'
            WHEN score > 452 THEN 'E'
            ELSE 'F'
        END AS risk_segment
    FROM scoring
),

scored_onboarded AS (
    SELECT
        o.user_id,
        o.onboarding_date,
        sc.risk_segment
    FROM onboarded_cohort o
    JOIN scorecard_1_0 sc
        ON o.user_id = sc.user_id
),

-- -- resignation
-- ordered_repayments AS (
--     SELECT
--         user_id,
--         vendor_id,
--         repaid_at,
--         LEAD(vendor_id) OVER(
--             PARTITION BY user_id
--             ORDER BY repaid_at
--         ) AS next_vendor
--     FROM tendopay_raw.customer_repayment_responses
--     WHERE vendor_id IN ('TPSD','TPFP')
-- ),

-- resigned_users AS (
--     SELECT
--         user_id,
--         MAX(DATE_ADD(DATE(repaid_at), INTERVAL 1 MONTH)) AS resignation_date
--     FROM ordered_repayments
--     WHERE vendor_id='TPSD'
--       AND next_vendor='TPFP'
--     GROUP BY user_id
-- ),

credit_limit AS (
    SELECT
        user_id,
        SAFE_CAST(value AS FLOAT64) AS credit_limit
    FROM prj-prod-dataplatform.tendopay_raw.customer_credit_information
    WHERE key='credit-limit'
),

oop_data AS (
    SELECT
        ee_customer_id AS user_id,
        osbal_as_of_resignation_date,
        osbal_as_of_oop_eligible_date,
        target_dev
    FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_feb26_20260301_oop_with_osbal`
),
oop_target AS (
  SELECT
    user_id,
    `target`, target_maturity_flag,ee_resignation_date_correct
  FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
  WHERE target_maturity_flag = 1

)

SELECT
format_date('%Y-%m',s.onboarding_date) as onboarding_month,

--s.risk_segment,


SUM(cl.credit_limit) AS tot_cl_amt_onboarded_with_1_0,



SUM( case when date(tgt.ee_resignation_date_correct) <= DATE '2026-03-01'
    THEN oop.osbal_as_of_resignation_date end)
    AS tot_os_principal_amt_as_of_resignation_date,


SUM(CASE
    WHEN   target_maturity_flag = 1
    THEN oop.osbal_as_of_oop_eligible_date end)
    AS tot_os_principal_amt_as_of_oop_eligible_dt,


SUM(
   CASE
    WHEN  tgt.target = 0
    THEN oop.osbal_as_of_oop_eligible_date
    END
) AS tot_os_principal_amt_from_bad_customers,


safe_divide(
SUM(
   CASE
    WHEN  tgt.target = 0
    THEN oop.osbal_as_of_oop_eligible_date
    END
),
SUM( case when date(tgt.ee_resignation_date_correct) <= DATE '2026-03-01'
    THEN oop.osbal_as_of_resignation_date end)

) *100 as unit_rate_of_all_who_left_job


FROM scored_onboarded s
JOIN oop_data r
    ON s.user_id = r.user_id  
join   oop_target tgt
on  cast(r.user_id as string) = tgt.user_id

LEFT JOIN credit_limit cl
    ON s.user_id = cl.user_id
LEFT JOIN oop_data oop
    ON s.user_id = oop.user_id

GROUP BY -- s.risk_segment, 
onboarding_month
ORDER BY -- s.risk_segment,
onboarding_month;
"""
dfv1 = client.query(sq).to_dataframe()
dfv1


,onboarding_month,tot_cl_amt_onboarded_with_1_0,tot_os_principal_amt_as_of_resignation_date,tot_os_principal_amt_as_of_oop_eligible_dt,tot_os_principal_amt_from_bad_customers,unit_rate_of_all_who_left_job
0,2025-06,18682278.5,7.162083e+06,6.183185e+06,4.321990e+06,60.345433
1,2025-07,16277842.0,6.001128e+06,5.348508e+06,3.452267e+06,57.526973
2,2025-08,11917418.0,4.034643e+06,3.716463e+06,2.736941e+06,67.835998
3,2025-09,18332580.5,9.283857e+06,9.245609e+06,7.025056e+06,75.669579
4,2025-10,25594467.5,1.540298e+07,1.553248e+07,1.216104e+07,78.952538
5,2025-11,13726164.0,8.242921e+06,8.704355e+06,6.714500e+06,81.457770
6,2025-12,5592677.5,2.242303e+06,3.144302e+06,1.847734e+06,82.403401


In [207]:
dfv1.to_excel(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\20260330_v1pesorate.xlsx", sheet_name='v1pesorate')

# Save file

In [247]:
dfs = {
    "slide2": df_summary,
    "slide3_upperpart": newprodusersprodscorejuneaug25,
    "slide3_lowerpart": newprodusersbsscorejuneaug25,
    "slide4_upperpart": extproduserprodscorejuly,
    "slide4_lowerpart": extprodusersbsscorejuly25,
    "slide5_upperpart": attrnewprodusersprodscorejuneaug25,
    "slide5_lowerpart": attrnewprodusersbsscorejuneaug25,
    "slide6_upperpart": attrextprodusersprodscorejuly25,
    "slide6_lowerpart": attrextproduserbsscorejuly25,
    "slide7_upperpart": df2v1,
    "slide7_lowerpart": data2,
    "slide8_upperpart": dfv1,
    "slide8_lowerpart": data4,
}

with pd.ExcelWriter(r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\tendomodelmonitoring_slidedata.xlsx", engine='openpyxl') as writer:
    for sheet_name, dataframe in dfs.items():
        dataframe.to_excel(writer, sheet_name=sheet_name[:31], index=False)

In [233]:
import pandas as pd
from datetime import datetime

# Create timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Build file name with timestamp
output_file = fr"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\tendomodelmonitoring_slidedata_{timestamp}.xlsx"

dfs = {
    "slide2": df_summary,
    "slide3_upperpart": newprodusersprodscorejuneaug25,
    "slide3_lowerpart": newprodusersbsscorejuneaug25,
    "slide4_upperpart": extproduserprodscorejuly,
    "slide4_lowerpart": extprodusersbsscorejuly25,
    "slide5_upperpart": attrnewprodusersprodscorejuneaug25,
    "slide5_lowerpart": attrnewprodusersbsscorejuneaug25,
    "slide6_upperpart": attrextprodusersprodscorejuly25,
    "slide6_lowerpart": attrextproduserbsscorejuly25,
    "slide7_upperpart": df2v1,
    "slide7_lowerpart": data2,
    "slide8_upperpart": dfv1,
    "slide8_lowerpart": data4,
}

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, dataframe in dfs.items():
        dataframe.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"✅ File saved: {output_file}")

✅ File saved: D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data\tendomodelmonitoring_slidedata_20260331_151911.xlsx
